# LifePolicy AI — Issue #5 초보자용 실습 노트북

**청킹 → 임베딩 → Qdrant → Neo4j → Hybrid 검색**을 직접 실행하면서 이해하기

- 브랜치: `feature/5-hybrid-rag-db` (담당: 주민혁 — 임베딩·DB)
- 참고 강의: `2. Qdrant.ipynb`(Qdrant VectorDB), `4. Hybrid RAG.ipynb`(GraphRAG with Neo4j)
- 구성: **작은 예제 → 실제 프로젝트 코드 → 실제 조회(읽기 전용) → 결과 해석**

> 이 노트북은 **학습용**입니다. 실제 DB에 쓰지 않고, 패키지를 설치하지 않고, 유료 LLM API를 부르지 않습니다.
> 숫자는 실제 코드 실행 결과이거나, 저장된 실험 결과 파일(`docs/development/experiments/`)에서 읽은 값입니다.

## 이 노트북 사용법

### 실행 전제조건 (장별)

| 장 | 필요한 것 | 없으면 |
|---|---|---|
| 0~3, 8(오프라인 부분), 11, 13, 14 | `.venv`, 프로젝트 파일 | — (항상 실행 가능) |
| 4·6·10·12의 임베딩 셀 | Ollama 실행 + `qwen3-embedding:0.6b` + `RUN_OLLAMA = True` | 안내 메시지만 출력 |
| 5·6의 Qdrant 조회 | Docker의 Qdrant 컨테이너 실행 + `RUN_DB_READ = True` | 안내 메시지만 출력 |
| 7~10의 Neo4j 조회 | Docker의 Neo4j 컨테이너 실행 + **적재된 데이터** | 안내 메시지만 출력 |

### 안전 규칙
- DB 조회는 모두 **읽기 전용**입니다. Neo4j는 읽기 세션(`ReadOnlyTimedRunner`)만, Qdrant는 count/scroll/search만 씁니다.
- Qdrant 저장 원리 실습은 **메모리 모드(`:memory:`)**에서만 합니다. 실제 컬렉션은 바꾸지 않습니다.
- `.env`의 비밀번호·API 키는 출력하지 않습니다.
- 임베딩은 `RUN_OLLAMA = True`로 직접 바꿨을 때만 실행됩니다. 전체 정책 재임베딩은 하지 않습니다(노트북 CPU에서는 매우 느림).

### VS Code에서 실행하기
- 커널은 프로젝트 `.venv`를 선택합니다.
- `.venv`에 `ipykernel`이 없으면 VS Code가 설치를 묻습니다. **설치 여부는 사용자가 직접 결정**합니다(이 노트북은 자동 설치하지 않습니다).

## 준비 1. 프로젝트 루트 찾기

**이번 셀의 목표:** 노트북이 어느 폴더에서 열리든 프로젝트 코드(`database/`)를 import할 수 있게 합니다.
- 사용하는 개념: `sys.path`(파이썬이 모듈을 찾는 폴더 목록), `pathlib.Path`
- 입력 데이터: 현재 작업 폴더
- 핵심 함수: `Path.cwd()`, `Path.parent`
- 예상 출력: `프로젝트 루트: SKN36-3rd-1Team` (폴더 이름)
- 왜 실행하나요?: 이 노트북은 `docs/development/`에 있어서, 그냥 `import database`를 하면 찾지 못합니다.

In [ ]:
import sys
import json
import math
import hashlib
from pathlib import Path

# 현재 폴더에서 위로 올라가며 AGENTS.md가 있는 폴더를 찾는다 → 그 폴더가 프로젝트 루트
ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    raise RuntimeError("프로젝트 루트(AGENTS.md가 있는 폴더)를 찾지 못했습니다. 노트북을 프로젝트 안에서 여세요.")

# 프로젝트 루트를 모듈 검색 경로 맨 앞에 넣는다 → `from database import ...`가 가능해진다
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("프로젝트 루트:", ROOT.name)   # 전체 경로 대신 폴더 이름만 출력 (개인 경로 노출 방지)

## 준비 2. 실행 스위치

**이번 셀의 목표:** 무거운 작업과 서버 접속을 **직접 켤 때만** 실행되게 합니다.
- `RUN_OLLAMA`: `True`로 바꾸면 Ollama로 짧은 문장 몇 개만 실제 임베딩합니다(질문 1~3개, 저장 없음).
- `RUN_DB_READ`: `True`이면 Qdrant·Neo4j가 켜져 있을 때 **읽기 전용** 조회를 합니다. 꺼져 있으면 안내만 합니다.
- 예상 출력: 현재 스위치 값

In [ ]:
RUN_OLLAMA = False   # ← 임베딩 실습을 하려면 True로 바꾸고 해당 셀을 다시 실행
RUN_DB_READ = True   # ← 서버가 꺼져 있으면 자동으로 건너뛴다 (오류 대신 안내 메시지)

print("RUN_OLLAMA  =", RUN_OLLAMA)
print("RUN_DB_READ =", RUN_DB_READ)

## 준비 3. 표 출력 도우미

**이번 셀의 목표:** 결과를 표처럼 보기 좋게 출력하는 작은 함수를 만듭니다. (pandas 없이 동작)
- 입력: dict의 리스트, 보여 줄 열 이름
- 예상 출력: 없음 (함수만 정의)

In [ ]:
def print_table(rows, columns, width=40):
    """rows(dict 리스트)를 columns 순서대로 표처럼 출력한다. 긴 글자는 width에서 자른다."""
    def cell(value):
        text = "" if value is None else str(value).replace("\n", " ").replace("\r", " ")
        return text if len(text) <= width else text[: width - 1] + "…"
    table = [[cell(column) for column in columns]] + [[cell(row.get(column)) for column in columns] for row in rows]
    sizes = [max(len(line[i]) for line in table) for i in range(len(columns))]
    for index, line in enumerate(table):
        print(" | ".join(value.ljust(sizes[i]) for i, value in enumerate(line)))
        if index == 0:
            print("-+-".join("-" * size for size in sizes))

print_table([{"이름": "예시", "값": 1}], ["이름", "값"])

---
# Chapter 0. 전체 서비스 이해

## 0-1. LifePolicy AI는 무엇을 만들려고 하나요?

정부 정책은 기관·사이트마다 흩어져 있고, 자격 조건이 공고문 속 긴 문장으로 쓰여 있어서 "나에게 해당하는지" 알기 어렵습니다.
LifePolicy AI는 **사용자의 프로필과 질문**을 받아 **관련 정책 + 공식 원문 근거 + 출처**를 돌려주는 서비스를 목표로 합니다.

| 구성 요소 | 하는 일 | 예시 |
|---|---|---|
| 사용자 프로필·질문 | 입력 | 29세, 서울, 신혼부부, 무주택 / "받을 수 있는 주거 지원이 있나요?" |
| 정책 데이터 | 공식 사이트에서 수집·정제 (장반석님 담당) | 정책 237개, 문서 246개 |
| **Qdrant** (VectorDB) | 원문 조각(청크)을 **의미**로 검색 | "전세자금 금리" → 금리 문단 |
| **Neo4j** (GraphDB) | 정책·기관·지역·조건의 **관계** 저장 | 정책 → 운영기관, 정책 → 조건 그룹 |
| **Hybrid 검색** | 두 결과를 `policy_id`로 결합 (**이번 작업의 핵심**) | 원문 근거 + 정책 문맥 |
| 자격 조건 확인 | 프로필로 조건 충족 여부 확인 (**강민지님 담당**) | 연령·지역·신청기간 |
| LLM 답변 생성 | 근거로 답변 작성 (**나지인·박소희님 담당**) | "~정책이 있으며 출처는 …" |

## 0-2. 전체 구조 다이어그램

```
[사용자 프로필 + 질문]
        │
        ▼
 ① 질문 임베딩 (Ollama, qwen3-embedding:0.6b)          ← 구현됨 (database/embedding.py)
        │
        ▼
 ② Qdrant 검색: 질문과 의미가 가까운 원문 청크 Top-K     ← 구현됨 (database/qdrant/store.py)
        │  청크마다 policy_id, document_id, 원문 위치
        ▼
 ③ Neo4j 조회: 그 policy_id의 기관·지역·조건 구조       ← 구현됨 (database/hybrid_search.py)
        │
        ▼
 ④ 결합: 정책별 {원문 근거 evidence + 그래프 문맥 graph} ← 구현됨 (search_policy_context)
        │
        ├──▶ ⑤ 자격 조건 확인 (강민지)                  ← 다른 팀원 영역, 이 노트북에서는 가상 흐름
        └──▶ ⑥ LLM 답변·출처 (나지인·박소희)            ← 다른 팀원 영역, 이 노트북에서는 가상 흐름
```

| 구분 | 내용 |
|---|---|
| **이번에 구현한 것** | 청킹(A/B1/B2), 구조 인식, 임베딩, Qdrant 실험 컬렉션(정책 5개, 107 point), Neo4j 적재·검증(노드 1,034 / 관계 3,615), Hybrid 검색(①~④) |
| **아직 없는 것** | 자격 판정, LLM 답변, 전체 정책 색인(246개 문서), 평가 질문 세트 기반 성능 평가 |

**이번 셀의 목표:** 프로젝트 파일이 실제로 있는지 확인합니다.
- 사용하는 개념: 파일 경로
- 입력 데이터: 모듈 경로 목록
- 핵심 함수·매개변수: `Path.exists()`
- 예상 출력: 각 파일 옆에 `있음`
- 왜 실행하나요?: 노트북이 설명하는 코드가 실제로 있는지 먼저 확인하기 위해서

In [ ]:
modules = {
    "database/handoff_data.py": "인계 데이터 읽기·검증",
    "database/chunking.py": "청킹 A/B1/B2·구조 인식",
    "database/embedding.py": "Ollama 임베딩",
    "database/qdrant/store.py": "Qdrant 저장·검색",
    "database/vector_index.py": "최소 적재·검색 흐름",
    "database/neo4j/load_graph.py": "Neo4j 안전 로더·검증",
    "database/hybrid_search.py": "Qdrant + Neo4j Hybrid 검색",
}
for path, role in modules.items():
    status = "있음" if (ROOT / path).exists() else "없음"
    print(f"{status:3} {path:32} {role}")

### Chapter 0 정리
- LifePolicy AI = 프로필·질문 → 관련 정책 + 원문 근거 + 출처
- 내 담당은 ①~④(임베딩·Qdrant·Neo4j·결합)이고, ⑤ 자격 판정과 ⑥ 답변 생성은 다른 팀원 영역이다.
- 현재 Qdrant에는 **정책 5개만** 색인되어 있다(스모크 검증용).

**문제 0-1 (내 말로 설명하기):** "Qdrant와 Neo4j를 둘 다 쓰는 이유"를 한 문장으로 말해 보세요.
**문제 0-2 (예상하기):** 위 셀에서 `없음`이 하나라도 나오면 어떤 상황일까요?
(정답은 마지막 장 「정답 및 해설」에 있습니다.)

---
# Chapter 1. RAG, VectorDB, GraphDB 기초

## 1-1. 용어를 정책 예시로 이해하기

| 용어 | 쉬운 설명 | 정책 예시 |
|---|---|---|
| **RAG** | 답하기 전에 관련 자료를 **찾아서(Retrieval)** 그 자료를 보고 **답을 만든다(Generation)** | "청년 월세 지원 금액?" → 공고문 문단을 찾아 그 내용으로 답 |
| **문서(Document)** | 정책 공고문·상세 페이지 하나 | `DOC_YC_003` (2026 우수인재 연계 기업인턴십) |
| **청크(Chunk)** | 문서를 검색하기 좋게 자른 조각 | `[지원내용] 콘텐츠 기업 대상 인턴 채용시 1인당 월 108만원…` |
| **임베딩(Embedding)** | 글을 숫자 목록(벡터)으로 바꾸는 것. 뜻이 비슷하면 숫자 방향도 비슷 | "월세 지원" ↔ "임차료 지원"이 가까운 벡터 |
| **벡터(Vector)** | 숫자 목록. 우리 모델은 숫자 **1024개** | `[0.012, -0.034, …]` |
| **코사인 유사도** | 두 벡터의 **방향**이 얼마나 비슷한지 (1에 가까울수록 비슷) | 질문 벡터 vs 청크 벡터 |
| **VectorDB** | 벡터를 저장하고 "가장 비슷한 벡터"를 빨리 찾는 DB (Qdrant) | 질문과 비슷한 청크 Top-5 |
| **GraphDB** | 대상(노드)과 관계(선)를 저장하는 DB (Neo4j) | 정책 —운영기관→ 기관 |

## 1-2. 작은 예제: 벡터와 코사인 유사도 (DB 없이)

**이번 셀의 목표:** 코사인 유사도가 무엇인지 숫자로 직접 계산해 봅니다.
- 사용하는 개념: 벡터, 내적, 길이(노름), 코사인 유사도 = 내적 ÷ (길이 × 길이)
- 입력 데이터: 가상 정책 문장 3개 (sample_only)
- 핵심 함수: 직접 만든 `toy_vector`, `cosine`
- 예상 출력: 비슷한 문장끼리 더 큰 점수
- 주의: 여기서 쓰는 "낱말 세기 벡터"는 **진짜 임베딩이 아닙니다.** 원리만 보기 위한 장난감입니다. 진짜 임베딩은 Chapter 4.

In [ ]:
# 가상 문장 (sample_only) — 실제 정책 내용이 아님
sentences = {
    "A": "청년 월세 지원 금액 안내",
    "B": "청년 주거 월세 지원 신청",
    "C": "신혼부부 전세 대출 금리",
}
vocabulary = ["청년", "월세", "지원", "신혼부부", "대출", "금리"]   # 장난감 벡터의 칸(차원) 이름

def toy_vector(text):
    """각 낱말이 문장에 몇 번 나오는지 세서 숫자 목록을 만든다 (진짜 임베딩 아님)."""
    return [text.count(word) for word in vocabulary]

def cosine(a, b):
    """코사인 유사도 = 내적 / (|a| × |b|). 1이면 방향이 같고 0이면 관련 없음."""
    dot = sum(x * y for x, y in zip(a, b))
    norm = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return dot / norm if norm else 0.0

vectors = {name: toy_vector(text) for name, text in sentences.items()}
for name, vector in vectors.items():
    print(name, vector, sentences[name])

In [ ]:
# 문장 쌍마다 코사인 유사도 계산
for left, right in [("A", "B"), ("A", "C"), ("B", "C")]:
    print(f"{left}-{right}: {cosine(vectors[left], vectors[right]):.3f}")
# A와 B는 "청년·월세·지원"을 공유 → 점수가 높고, C는 공유 낱말이 없어 0
# 주의: A와 B가 1.000(완전히 같음)으로 나오는 이유는 두 문장의 다른 낱말("금액·안내" vs "주거·신청")이
#       vocabulary에 없어서 벡터가 똑같아졌기 때문이다. 낱말 세기 방식은 목록에 없는 낱말·비슷한 뜻을 모른다.
#       진짜 임베딩 모델은 낱말 목록 없이 문장 전체의 뜻을 숫자로 바꾼다 (Chapter 4).

## 1-3. 작은 예제: 그래프 (노드·레이블·관계·속성)

Neo4j 용어:
- **노드(Node)**: 대상 하나. 예) 정책 하나, 기관 하나
- **레이블(Label)**: 노드의 종류. 예) `Policy`, `Organization`
- **관계(Relationship)**: 노드 사이의 방향 있는 선. 예) `(Policy)-[:ADMINISTERED_BY]->(Organization)`
- **속성(Property)**: 노드·관계에 붙은 값. 예) `policy_name`, `review_status`
- **출발 레이블 / 도착 레이블**: 관계 화살표가 **시작하는** 노드의 레이블과 **도착하는** 노드의 레이블.
  `ADMINISTERED_BY`의 출발은 `Policy`, 도착은 `Organization`

**이번 셀의 목표:** 파이썬 dict로 아주 작은 그래프를 만들고 "이 정책의 운영기관"을 찾아봅니다. (가상 데이터, sample_only)

In [ ]:
# 가상 그래프 (sample_only)
toy_nodes = [
    {"label": "Policy", "id": "POL_S1", "policy_name": "가상 청년 월세 지원"},
    {"label": "Policy", "id": "POL_S2", "policy_name": "가상 신혼부부 전세 대출"},
    {"label": "Organization", "id": "AGENCY_S1", "name": "가상 시청"},
    {"label": "Organization", "id": "AGENCY_S2", "name": "가상 주택기금"},
]
toy_relationships = [
    {"source": "POL_S1", "type": "ADMINISTERED_BY", "target": "AGENCY_S1", "review_status": "sample_only"},
    {"source": "POL_S2", "type": "ADMINISTERED_BY", "target": "AGENCY_S2", "review_status": "sample_only"},
]

def find_agency(policy_id):
    """정책 → (ADMINISTERED_BY) → 기관을 따라가서 기관 이름을 돌려준다."""
    for rel in toy_relationships:
        if rel["source"] == policy_id and rel["type"] == "ADMINISTERED_BY":
            return next(node["name"] for node in toy_nodes if node["id"] == rel["target"])
    return None

print(find_agency("POL_S1"))
print(find_agency("POL_S2"))

## 1-4. 세 가지 ID: policy_id, document_id, chunk_id

| ID | 무엇 | 예시 | 어디에 |
|---|---|---|---|
| `policy_id` | 정책 하나 | `POL_YC_003` | Neo4j `Policy` 노드, Qdrant payload |
| `document_id` | 정책의 공식 문서 하나 (정책 1개에 문서 여러 개 가능) | `DOC_YC_003` | Neo4j `Document` 노드, Qdrant payload |
| `chunk_id` | 문서를 자른 조각 하나 | `CHK_B2_DOC_YC_003_C001` | Qdrant point (`point_id = uuid5(chunk_id)`) |

**이번 셀의 목표:** 실제 프로젝트 함수 `make_chunk_id`, `make_point_id`가 ID를 어떻게 만드는지 봅니다. (DB 없이)
- 핵심 함수: `chunking.make_chunk_id(document_id, seq, version)`, `chunking.make_point_id(chunk_id)`
- 예상 출력: `CHK_B2_DOC_YC_003_C001`과 UUID 문자열. **같은 chunk_id는 항상 같은 UUID**

In [ ]:
from database import chunking as ck

chunk_id = ck.make_chunk_id("DOC_YC_003", 1, "B2")
print("chunk_id :", chunk_id)
print("point_id :", ck.make_point_id(chunk_id))
print("다시 계산 :", ck.make_point_id(chunk_id))   # 같은 입력 → 같은 UUID (uuid5) → 재적재해도 중복되지 않는 이유

### Chapter 1 정리
- 임베딩 = 글 → 숫자 목록(벡터), 코사인 유사도 = 벡터 방향의 비슷함
- VectorDB는 "의미가 비슷한 글"을, GraphDB는 "무엇과 무엇이 어떻게 연결되었나"를 찾는 데 강하다.
- 관계에는 방향이 있고, 출발·도착 레이블이 정해져 있다.
- `point_id = uuid5(chunk_id)` → 같은 청크는 항상 같은 ID

**문제 1-1 (예상하기):** `cosine([1, 0], [0, 1])`의 값은?
**문제 1-2 (빈칸 채우기):** 아래 셀의 `...`을 채워 `POL_S2`의 기관 이름이 나오게 하세요.

In [ ]:
# 문제 1-2: ... 을 알맞은 정책 ID 문자열로 바꾸세요
answer_policy_id = ...
if answer_policy_id is ...:
    print("빈칸을 채워 보세요")
else:
    print(find_agency(answer_policy_id))

---
# Chapter 2. 정책 원본 데이터 읽기

## 2-1. 데이터 폴더 구조

| 단계 | 위치 | 내용 | 담당 |
|---|---|---|---|
| 원본 | `data/raw/` | 공식 PDF·HTML·API 응답 원본. **Git에 올리지 않아 로컬에 없을 수 있음** | 장반석 |
| 정제 | `data/processed/` | `policies.jsonl`(정책 237), `documents.jsonl`(문서 246, 본문 `text`·`content_hash`), `relationships.jsonl` | 장반석 |
| DB 인계 | `data/handoff/` | Neo4j 노드 9종·관계(`neo4j/`), Qdrant 청크 557개(`qdrant/`), `manifest.json`(체크섬) | 장반석 |
| 메타데이터 | `data/metadata/` | API 목록, ID 매핑 등 | 장반석 |

> 이 폴더들은 **장반석님 담당 파일**이므로 읽기만 합니다.

**이번 셀의 목표:** 정제 문서 246개를 읽고, 본문 해시가 맞는지 검사합니다.
- 사용하는 개념: JSONL(한 줄에 JSON 하나), SHA-256 해시
- 입력 데이터: `data/processed/documents.jsonl`
- 핵심 함수·매개변수: `handoff_data.load_documents(verify_hash=True)`
- 예상 출력: `문서 수: 246`
- 왜 실행하나요?: 모든 이후 단계(청킹·임베딩)의 입력이 이 문서들이기 때문에

In [1]:
from database import handoff_data as hd

documents = hd.load_documents(verify_hash=True)   # sha256(text) != content_hash 이면 오류를 낸다
print("문서 수:", len(documents))
print("첫 문서 필드:", sorted(documents[0].keys()))

ModuleNotFoundError: No module named 'database'

**이번 셀의 목표:** 문서 1건(DOC_YC_003)의 필드를 살펴봅니다.
- 사용하는 개념: dict 필드
- 입력 데이터: documents 리스트
- 핵심 함수·매개변수: dict 조회
- 예상 출력: 문서 ID·정책 ID·출처·본문 앞부분
- 왜 실행하나요?: 각 필드가 무엇을 뜻하는지 알아야 청크·payload를 이해할 수 있어서

In [ ]:
doc = next(d for d in documents if d["document_id"] == "DOC_YC_003")
for field in ["document_id", "policy_id", "title", "source_type", "source_url", "retrieved_at",
              "review_status", "verification_status", "content_hash"]:
    print(f"{field:20}: {doc.get(field)}")
print("본문 길이:", len(doc["text"]), "자")
print(doc["text"][:300])

**필드 설명**
- `document_id` / `policy_id`: 문서 ID와 그 문서가 속한 정책 ID
- `source_url`·`retrieved_at`: 출처와 수집 시각 → 답변에 출처를 표시할 때 사용
- `content_hash`: 본문(`text`)의 SHA-256 지문. 본문이 한 글자라도 바뀌면 값이 달라진다 → **변경 감지·무결성 확인**
- `review_status`·`verification_status`: 수집 데이터 검증 상태 (`verified`, `extracted` 등)

**이번 셀의 목표:** content_hash가 무엇인지 직접 계산해 봅니다.
- 사용하는 개념: SHA-256
- 입력 데이터: DOC_YC_003 본문
- 핵심 함수·매개변수: `hashlib.sha256(text.encode('utf-8')).hexdigest()`
- 예상 출력: `일치: True`
- 왜 실행하나요?: 해시로 원문 변경을 감지하는 원리를 확인

In [ ]:
recomputed = hashlib.sha256(doc["text"].encode("utf-8")).hexdigest()
print("저장된 해시 :", doc["content_hash"][:16], "…")
print("다시 계산   :", recomputed[:16], "…")
print("일치:", recomputed == doc["content_hash"])
print("한 글자 바꾸면:", hashlib.sha256((doc["text"] + " ").encode("utf-8")).hexdigest()[:16], "…")

**이번 셀의 목표:** 정책 1건(POL_YC_003)을 정제 데이터에서 읽습니다.
- 사용하는 개념: 정책 레코드
- 입력 데이터: `data/processed/policies.jsonl`
- 핵심 함수·매개변수: `hd.read_jsonl(path)`
- 예상 출력: 정책명·분야·지역·자격 요약 등
- 왜 실행하나요?: 정책(Policy)과 문서(Document)가 다른 것임을 확인

In [ ]:
policies = hd.read_jsonl(ROOT / "data" / "processed" / "policies.jsonl")
policy = next(p for p in policies if p["policy_id"] == "POL_YC_003")
print("정책 수:", len(policies))
for field in ["policy_id", "policy_name", "category", "region", "life_stages", "target_groups",
              "eligibility", "benefit", "source_document_ids", "verification_status"]:
    print(f"{field:20}: {str(policy.get(field))[:90]}")

**이번 셀의 목표:** DB 인계 데이터(Neo4j 노드·관계, Qdrant 청크)를 읽어 개수를 셉니다.
- 사용하는 개념: 인계 패키지
- 입력 데이터: `data/handoff/neo4j/*.jsonl`, `data/handoff/qdrant/payload_chunks.jsonl`
- 핵심 함수·매개변수: `hd.load_neo4j_nodes()`, `hd.load_neo4j_relationships()`, `hd.load_payload_chunks()`
- 예상 출력: 노드 9종 합계 1,034 / 관계 3,615 / 청크 557
- 왜 실행하나요?: DB에 넣을 데이터의 실제 크기를 파일에서 직접 확인

In [ ]:
nodes_by_label = hd.load_neo4j_nodes()
handoff_relationships = hd.load_neo4j_relationships()
payload_chunks = hd.load_payload_chunks()

for label, rows in nodes_by_label.items():
    print(f"{label:22} {len(rows):5}")
print("노드 합계:", sum(len(rows) for rows in nodes_by_label.values()))
print("관계 수  :", len(handoff_relationships))
print("A안 청크 :", len(payload_chunks))

**이번 셀의 목표:** 정제 데이터의 관계 파일과 인계 데이터의 관계 파일을 비교합니다.
- 사용하는 개념: 데이터 단계별 차이
- 입력 데이터: processed/relationships.jsonl vs handoff/neo4j/relationships.jsonl
- 핵심 함수·매개변수: `collections.Counter`
- 예상 출력: 정제 2,537 / 인계 3,615 — 조건 구조 관계 3종이 인계에만 있음
- 왜 실행하나요?: 원본·정제·인계 데이터가 같은 것이 아님을 확인

In [ ]:
from collections import Counter

processed_relationships = hd.read_jsonl(ROOT / "data" / "processed" / "relationships.jsonl")
processed_types = Counter(r["relation"] for r in processed_relationships)
handoff_types = Counter(r["relation"] for r in handoff_relationships)
rows = [{"관계 타입": t, "정제": processed_types.get(t, 0), "인계": handoff_types[t]} for t in handoff_types]
print_table(rows, ["관계 타입", "정제", "인계"])
print("합계:", sum(processed_types.values()), "/", sum(handoff_types.values()))

**해석:** 인계 데이터에는 조건 구조를 표현하는 `HAS_ROOT_CONDITION_GROUP`·`CONTAINS_GROUP`·`CONTAINS_CONDITION`이 추가되어 있습니다.
Neo4j에는 **인계 데이터(3,615)**를 적재했습니다.

원본 폴더도 확인해 봅시다. 원본은 Git에 올리지 않기 때문에 로컬에 없을 수 있습니다.

In [ ]:
raw_dir = ROOT / "data" / "raw"
raw_files = [p for p in raw_dir.rglob("*") if p.is_file() and p.name != ".gitkeep"] if raw_dir.exists() else []
print("data/raw 폴더 있음:", raw_dir.exists(), "| 원본 파일 수:", len(raw_files))
# 0개여도 정상일 수 있다: 이후 단계는 정제 본문(documents.jsonl)의 text와 content_hash만 쓴다.

### Chapter 2 정리
- 정책(Policy) 237개 ≠ 문서(Document) 246개: 정책 하나에 문서가 여러 개일 수 있다.
- `content_hash`는 본문 지문이다. 같으면 내용이 같다는 뜻이다.
- 원본 → 정제 → 인계로 갈수록 DB에 맞는 모양으로 바뀐다(예: 조건 구조 관계 추가).
- 이 데이터는 팀원 담당이므로 읽기만 한다.

**문제 2-1 (예상하기):** 문서 본문 끝에 공백 하나를 더하면 `content_hash`는 같을까요?
**문제 2-2 (내 말로 설명하기):** 정제 관계(2,537)와 인계 관계(3,615)의 차이는 무엇인가요?

---
# Chapter 3. 청킹

## 3-1. 강사님 강의 방식

`2. Qdrant.ipynb`에서는 텍스트 파일을 다음처럼 나눴습니다.

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
splits = text_splitter.split_documents(documents)
```

- `chunk_size=200`: 조각 최대 200자, `chunk_overlap=50`: 앞 조각과 50자 겹침
- 구분자(문단 → 줄 → 띄어쓰기 → 글자) 순서로 자를 곳을 찾는 **길이 기반** 분할

## 3-2. 우리 프로젝트의 세 가지 방식

| 방식 | 기준 | 전체 청크 수 | 특징 |
|---|---|---|---|
| **A** | 인계 패키지의 고정 길이(750자, 80자 겹침) | 557 | 단순. 제목·자격 문단 경계를 무시하고 자를 수 있음 |
| **B1** | 원문의 **필드·제목 경계** (`[지원대상]`, `■ 입주자격` 등) | 1,300 | 주제가 섞이지 않음. 짧은 조각이 많음 |
| **B2** | B1에서 100자 미만 조각을 **허용된 의미 그룹끼리만** 병합 | 1,081 | 경계는 지키고 짧은 조각만 줄임 |

- 우리 환경에는 `langchain_text_splitters`가 없고, **원문 위치(`char_start`, `char_end`)를 정확히 기록**해야 해서 같은 원리를 표준 라이브러리로 직접 구현했습니다(`chunking.split_long_span`).
- **B2가 모든 질문에서 가장 좋다고 확정되지 않았습니다.** 실험에 쓴 선택일 뿐이고, 최종 청킹 규칙은 평가 후 팀이 정합니다(PRD §13).

**이번 셀의 목표:** 고정 길이 분할이 문장을 어떻게 자르는지 장난감으로 봅니다.
- 사용하는 개념: 고정 창(window)·겹침(overlap)
- 입력 데이터: 가상 정책 문장 (sample_only)
- 핵심 함수·매개변수: 직접 만든 `fixed_split(text, size, overlap)`
- 예상 출력: 단어 중간에서 잘린 조각
- 왜 실행하나요?: B1/B2를 왜 만들었는지 이해하기 위해

In [ ]:
sample_text = "[지원대상] 만 19세 이상 34세 이하 무주택 청년. [지원내용] 월 최대 20만원 임차료를 지원합니다. (sample_only)"

def fixed_split(text, size, overlap):
    """size 글자씩 자르고 다음 조각은 overlap 글자 앞에서 시작한다 (구분자를 보지 않는 가장 단순한 방식)."""
    pieces, start = [], 0
    while start < len(text):
        pieces.append(text[start:start + size])
        start += size - overlap
    return pieces

for i, piece in enumerate(fixed_split(sample_text, size=30, overlap=5), 1):
    print(i, repr(piece))
# [지원대상]과 [지원내용]이 한 조각에 섞이거나 단어가 잘리는 것을 볼 수 있다.

**이번 셀의 목표:** 실제 문서 1개(DOC_YC_003)를 A/B1/B2로 나눠 비교합니다.
- 사용하는 개념: 청킹 전략
- 입력 데이터: DOC_YC_003 문서, 인계 청크(A)
- 핵심 함수·매개변수: `vector_index.prepare_chunks(docs, strategy, payload_chunks)`
- 예상 출력: A 2개 / B1 4개 / B2 3개 (전략별 표)
- 왜 실행하나요?: 같은 문서가 전략에 따라 어떻게 달라지는지 직접 보기 위해

In [ ]:
from database import vector_index as vi

chunks_by_strategy = {strategy: vi.prepare_chunks([doc], strategy, payload_chunks) for strategy in ("A", "B1", "B2")}
for strategy, chunks in chunks_by_strategy.items():
    print(f"=== {strategy}: {len(chunks)}개")
    print_table([{"chunk_id": c["chunk_id"], "길이": len(c["text"]), "section_title": c.get("section_title"),
                  "char_start": c.get("char_start"), "char_end": c.get("char_end")} for c in chunks],
                ["chunk_id", "길이", "section_title", "char_start", "char_end"], width=45)

**해석 포인트**
- A의 `section_title`은 인계 패키지가 붙인 이름이고, B1/B2는 원문 필드 이름(`[지원내용]` 등)입니다.
- B2는 B1의 짧은 조각 일부를 **같은 주제 그룹끼리** 합친 결과입니다.

**이번 셀의 목표:** char_start·char_end로 원문을 복원해 청크 본문과 같은지 확인합니다.
- 사용하는 개념: 문자열 슬라이싱 `text[start:end]`
- 입력 데이터: B2 청크
- 핵심 함수·매개변수: 슬라이싱
- 예상 출력: 모두 `True`
- 왜 실행하나요?: 청크가 원문을 바꾸지 않았다는 것(원문 보존)을 직접 검증

In [ ]:
for chunk in chunks_by_strategy["B2"]:
    restored = doc["text"][chunk["char_start"]:chunk["char_end"]]   # 원문에서 그 위치를 그대로 잘라 본다
    print(chunk["chunk_id"], restored == chunk["text"])

**이번 셀의 목표:** B2가 어떤 B1 조각을 합쳤는지(`merged_from`) 봅니다.
- 사용하는 개념: 짧은 청크 병합
- 입력 데이터: DOC_BOK_001
- 핵심 함수·매개변수: `chunking.chunk_documents([doc], strategy='B2')`
- 예상 출력: 합쳐진 청크의 원래 B1 ID 목록
- 왜 실행하나요?: 병합이 허용된 그룹끼리만 일어난다는 것을 확인

In [ ]:
bok = next(d for d in documents if d["document_id"] == "DOC_BOK_001")
b2_chunks, _reports = ck.chunk_documents([bok], strategy="B2")
for chunk in b2_chunks:
    merged = chunk.get("merged_from")
    if merged:   # 병합된 청크만 출력
        print(chunk["chunk_id"], "| 길이", len(chunk["text"]), "| 그룹", chunk.get("section_groups"))
        print("   merged_from:", merged)

**이번 셀의 목표:** 전체 246개 문서의 전략별 청크 수를 계산합니다.
- 사용하는 개념: 전체 집계
- 입력 데이터: documents 전체
- 핵심 함수·매개변수: `ck.chunk_documents(documents, strategy)`
- 예상 출력: B1 1,300 / B2 1,081 (A는 인계 557)
- 왜 실행하나요?: 문서에 적힌 숫자를 직접 재현 (몇 초 걸림, DB 불필요)

In [ ]:
b1_all, _ = ck.chunk_documents(documents, strategy="B1")
b2_all, b2_reports = ck.chunk_documents(documents, strategy="B2")
print("A :", len(payload_chunks))
print("B1:", len(b1_all))
print("B2:", len(b2_all))

**이번 셀의 목표:** 원문 누락이 없는지 `verify_coverage`로 검사합니다.
- 사용하는 개념: 커버리지(원문이 빠짐없이 덮였는가)
- 입력 데이터: DOC_BOK_001과 그 B2 청크·처리 기록
- 핵심 함수·매개변수: `ck.verify_coverage(document, chunks, excluded_spans)`
- 예상 출력: `missing_chars: 0`, `unintended_overlap_chars: 0`
- 왜 실행하나요?: 청킹이 글자를 잃어버리지 않았는지 증명

In [ ]:
bok_report = next(r for r in b2_reports if r["document_id"] == "DOC_BOK_001")
coverage = ck.verify_coverage(bok, [c for c in b2_all if c["document_id"] == "DOC_BOK_001"], bok_report["excluded_spans"])
print({key: coverage[key] for key in ("missing_chars", "intentional_overlap_chars", "unintended_overlap_chars", "excluded_in_chunks")})

**참고: 문서 구조 인식(`analyze_structure`)** — 청킹 전에 "이 문서에 제목 구조가 있는가"를 판정합니다
(structured / partial / unstructured). 판정이 structured여도 **제목 인식이 정확하다는 보장은 아닙니다**(메뉴·요약 상자를 제목으로 오인한 사례가 있음).

In [ ]:
analysis = ck.analyze_structure(doc)
print("판정:", analysis["status"], "| 권장 방식:", analysis["recommended_method"])
print("인정된 제목:", [h["line"] for h in analysis["headings"]][:5])

### Chapter 3 정리
- 강의는 길이 기반(RecursiveCharacterTextSplitter), 우리는 **원문 구조 기반**(B1/B2) + 원문 위치 기록
- 모든 청크는 `text == 원문[char_start:char_end]`
- B2는 실험용 선택이며 최종 확정이 아니다(P3 같은 실패 사례가 있다 → Chapter 6).

**문제 3-1 (예상하기):** `doc["text"][0:10]`은 B2 첫 청크의 `text` 앞 10자와 같을까요? (힌트: char_start가 0인가?)
**문제 3-2 (빈칸 채우기):** 아래 셀의 `...`을 채워 B1 청크 수를 출력하세요.

In [ ]:
# 문제 3-2: ... 을 알맞은 전략 이름 문자열로 바꾸세요
strategy_name = ...
if strategy_name is ...:
    print("빈칸을 채워 보세요")
else:
    print(len(vi.prepare_chunks([doc], strategy_name, payload_chunks)))

---
# Chapter 4. 임베딩

## 4-1. 강의 방식 vs 우리 방식

| | 강의 (`2. Qdrant.ipynb`) | 우리 (`database/embedding.py`) |
|---|---|---|
| 코드 | `OllamaEmbeddings(model="qwen3-embedding:0.6b")` | `make_ollama_embedder()` → `ollama_embed_batch()` |
| 라이브러리 | `langchain_ollama` | `requests`로 Ollama `/api/embed` 직접 호출 |
| 검증 | 라이브러리에 맡김 | 응답 개수·**1024차원**·NaN/무한대 검사, 시간 초과·연결 실패를 `EmbeddingError`로 |
| 길이 초과 | — | `truncate: False` (긴 입력을 몰래 자르지 않고 오류로 알림) |
| 입력 | 청크 본문 | **임베딩 입력 규칙**으로 정책명·섹션명·본문을 조합 (저장 본문과 분리) |

- 모델 `qwen3-embedding:0.6b`: 벡터 1024차원, Ollama 서버에서 실행. **기준선(baseline)일 뿐 최종 모델이 아닙니다.**
- `0.6b`와 `4b`: 이 PC의 `ollama list` 기준 크기는 0.6b 639MB, 4b 2.5GB입니다. 4b는 더 크고 느리며 메모리를 더 씁니다.
  **두 모델의 검색 품질은 이 프로젝트에서 비교하지 않았습니다.**

**이번 셀의 목표:** 임베딩 입력 규칙 v1과 v2의 차이를 봅니다 (DB·Ollama 없이).
- 사용하는 개념: 임베딩 입력 vs 저장 본문
- 입력 데이터: DOC_YC_003의 B2 첫 청크, 정책 메타데이터
- 핵심 함수·매개변수: `em.build_embedding_text(chunk, policy_meta, rule)`
- 예상 출력: v1은 정책명·섹션명을 덧붙이고, v2는 본문에 이미 있는 줄은 생략
- 왜 실행하나요?: 같은 청크라도 규칙에 따라 벡터가 달라진다는 것을 이해

In [ ]:
from database import embedding as em

policy_metadata = hd.load_policy_metadata(payload_chunks)
chunk = chunks_by_strategy["B2"][0]
meta = policy_metadata[chunk["policy_id"]]
for rule in (em.RULE_POLICY_SECTION_BODY, em.RULE_POLICY_SECTION_BODY_V2, em.RULE_BODY_ONLY):
    text = em.build_embedding_text(chunk, meta, rule)
    print(f"[{rule}] 길이 {len(text)}")
    print("   앞부분:", text[:120].replace("\n", " ⏎ "))
print("저장 본문(text)은 그대로:", chunk["text"][:60].replace("\n", " ⏎ "))

**v2를 만든 이유:** v1에서는 B 청크의 약 83%에서 섹션명이 두 번 들어갔습니다(본문이 이미 `[지원내용] …`으로 시작).
v2는 본문의 **줄 시작**에 이미 있는 제목·정책명만 생략합니다. 규칙 이름은 Qdrant payload `embedding_input_rule`에 남아 다른 규칙의 벡터와 섞이지 않습니다.

**문서 임베딩 vs 질문 임베딩:** 문서는 위 규칙으로 만든 입력을, 질문은 `build_query_text(question, query_template)`로 만든 입력을 임베딩합니다.
Hybrid 검색 기본값은 `query_template=None`(질문 그대로)입니다.

In [ ]:
question = "콘텐츠 기업이 인턴을 채용하면 1인당 인건비를 얼마 지원받나요?"
print(em.build_query_text(question, None))
print(em.build_query_text(question, "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:{question}"))

## 4-2. (선택) 실제 임베딩 실습 — `RUN_OLLAMA = True`일 때만

**이번 셀의 목표:** 짧은 문장 3개를 실제로 임베딩해서 차원·길이·유사도를 확인합니다.
- 전제조건: Ollama 실행, `qwen3-embedding:0.6b` 설치 (모델을 자동으로 내려받지 않습니다)
- 핵심 함수: `em.make_ollama_embedder(batch_size, timeout)`, `em.vector_norm`
- 예상 출력: 차원 `1024`, 길이(노름) 약 `1.0`, 비슷한 문장끼리 더 높은 코사인 값
- 노트북 CPU에서는 처음 실행 시 모델 로딩으로 몇 초가 걸릴 수 있습니다.

In [ ]:
def ollama_ready():
    """Ollama 서버와 모델이 있는지 읽기만 해서 확인한다 (모델을 내려받지 않는다)."""
    import requests
    try:
        url = em.resolve_ollama_base_url()
        models = requests.get(url + "/api/tags", timeout=5).json().get("models", [])
        names = {m.get("name") for m in models}
        if em.DEFAULT_EMBEDDING_MODEL not in names:
            print(f"Ollama는 켜져 있지만 {em.DEFAULT_EMBEDDING_MODEL} 모델이 없습니다. (설치는 사용자가 직접 결정)")
            return False
        return True
    except Exception as error:
        print("Ollama에 접속하지 못했습니다:", type(error).__name__, "→ Ollama 앱을 실행한 뒤 다시 시도하세요.")
        return False

OLLAMA_OK = RUN_OLLAMA and ollama_ready()
print("임베딩 실습 실행:", OLLAMA_OK, "(RUN_OLLAMA =", RUN_OLLAMA, ")")

In [ ]:
if OLLAMA_OK:
    embed = em.make_ollama_embedder(batch_size=3, timeout=180)
    texts = ["청년 월세 지원 금액", "청년 임차료 지원 금액", "어선 감척 지원 요건"]   # sample_only
    real_vectors = embed(texts)
    print("벡터 개수:", len(real_vectors), "| 차원:", len(real_vectors[0]))
    print("길이(노름):", [round(em.vector_norm(v), 4) for v in real_vectors])
    print("앞 5개 숫자:", [round(x, 4) for x in real_vectors[0][:5]])
else:
    print("건너뜀: RUN_OLLAMA=True로 바꾸고 Ollama를 켠 뒤 실행하세요.")

In [ ]:
if OLLAMA_OK:
    for i in range(3):
        for j in range(i + 1, 3):
            print(f"{texts[i]} ↔ {texts[j]}: {cosine(real_vectors[i], real_vectors[j]):.4f}")
    # 실행 결과의 숫자는 모델·하드웨어에 따라 조금 다를 수 있다. 크기 비교(어느 쌍이 더 가까운가)를 본다.
else:
    print("건너뜀")

### Chapter 4 정리
- 임베딩 모델은 `qwen3-embedding:0.6b`(1024차원), Ollama `/api/embed`를 직접 호출한다.
- 저장 본문과 임베딩 입력을 분리했다(v2 규칙).
- `truncate=False`: 긴 입력을 몰래 자르지 않는다.
- 0.6b와 4b의 품질은 비교하지 않았다.

**문제 4-1 (예상하기):** 정규화된(길이 1) 벡터 두 개의 코사인 유사도는 무엇과 같을까요? (힌트: 내적)
**문제 4-2 (내 말로 설명하기):** 저장 본문(`text`)과 임베딩 입력을 왜 분리했나요?

---
# Chapter 5. Qdrant에 데이터 저장

## 5-1. 강의 방식 vs 우리 방식

| | 강의 | 우리 (`database/qdrant/store.py`) |
|---|---|---|
| 저장 | `QdrantVectorStore.from_documents(documents=splits, embedding=embeddings, location="http://localhost:6333", collection_name="rag_keywords")` | `ensure_collection()` + `build_points()` + `upsert_points()` (QdrantClient 직접 사용) |
| metadata 위치 | LangChain이 `metadata.*` 아래에 저장 → 필터 키가 `metadata.category` | payload **최상위**에 저장 → 필터 키가 `document_id` |
| point ID | `ids`를 넘기지 않으면 라이브러리가 새 UUID를 만든다 → 다시 넣으면 중복될 수 있음 | `point_id = uuid5(chunk_id)` 고정 → **재적재해도 중복 0** |
| 주소 | `localhost` | `127.0.0.1` 권장 (Windows에서 localhost는 요청마다 약 2초 지연을 실측) |
| 안전장치 | — | `exp_`로 시작하는 실험 컬렉션만 허용, 서비스 이름 `lifepolicy_chunks` 거부, 삭제 함수 없음 |

**용어**
- **Collection**: 벡터를 담는 "표" (`exp_b2_qwen3emb06b_v1`)
- **Point**: 표의 한 행 = 청크 하나 = `id` + `vector` + `payload`
- **Payload**: point에 붙은 메타데이터 dict (정책 ID, 원문, 위치, 출처…)
- **Upsert**: 같은 id가 있으면 덮어쓰고 없으면 추가 (update + insert)
- **Payload Index**: 필터 검색을 빠르게 하려고 특정 payload 필드에 만드는 색인
- **Cosine**: 벡터 비교 방식 (방향의 비슷함)

**이번 셀의 목표:** 메모리 모드 Qdrant에서 저장 원리를 실습합니다 (실제 DB와 무관).
- 사용하는 개념: Collection·Point·Upsert
- 입력 데이터: 가상 4차원 벡터 2개 (sample_only)
- 핵심 함수·매개변수: `store.get_client(location=':memory:')`, `store.ensure_collection`, `store.upsert_points`
- 예상 출력: 두 번 upsert해도 point 수 2
- 왜 실행하나요?: 실제 컬렉션을 건드리지 않고 저장·중복 방지 원리를 확인

In [ ]:
import logging
from database.qdrant import store

logging.getLogger().setLevel(logging.ERROR)    # 메모리 모드의 "payload index는 효과 없음" 경고를 숨긴다

memory_client = store.get_client(location=":memory:")          # 이 노트북 프로세스 안에서만 사는 임시 Qdrant
demo_collection = "exp_notebook_demo_v1"                       # 실험용 이름 규칙(exp_...)을 따른다
print(store.ensure_collection(memory_client, demo_collection, dim=4))   # 없으면 만든다 → "created"

In [ ]:
demo_points = [
    {"id": ck.make_point_id("CHK_DEMO_DOC_S1_C001"), "vector": [0.9, 0.1, 0.0, 0.0],
     "payload": {"chunk_id": "CHK_DEMO_DOC_S1_C001", "policy_id": "POL_S1", "document_id": "DOC_S1",
                 "text": "가상 청년 월세 지원 (sample_only)"}},
    {"id": ck.make_point_id("CHK_DEMO_DOC_S2_C001"), "vector": [0.0, 0.1, 0.9, 0.0],
     "payload": {"chunk_id": "CHK_DEMO_DOC_S2_C001", "policy_id": "POL_S2", "document_id": "DOC_S2",
                 "text": "가상 신혼부부 전세 대출 (sample_only)"}},
]
store.upsert_points(memory_client, demo_collection, demo_points)
print("1차 upsert 후:", store.count_points(memory_client, demo_collection))
store.upsert_points(memory_client, demo_collection, demo_points)   # 같은 id로 다시 → 덮어쓰기
print("2차 upsert 후:", store.count_points(memory_client, demo_collection))   # 늘지 않는다

In [ ]:
# 가상 질문 벡터 [1, 0, 0, 0]과 가장 가까운 point 찾기
for hit in store.search_chunks(memory_client, demo_collection, [1.0, 0.0, 0.0, 0.0], k=2):
    print(hit["rank"], round(hit["score"], 4), hit["policy_id"], hit["payload"]["text"])

**이번 셀의 목표:** 보호 규칙을 확인합니다.
- 사용하는 개념: 안전장치
- 입력 데이터: 서비스 컬렉션 이름
- 핵심 함수·매개변수: `store.assert_experiment_collection(name)`
- 예상 출력: `CollectionGuardError` 메시지
- 왜 실행하나요?: 실수로 서비스 컬렉션을 건드리지 않도록 막는 장치를 확인

In [ ]:
for name in ["exp_b2_qwen3emb06b_v1", "lifepolicy_chunks"]:
    try:
        print("허용:", store.assert_experiment_collection(name))
    except store.CollectionGuardError as error:
        print("거부:", error)

## 5-2. 실제 실험 컬렉션 읽기 (읽기 전용)

> 현재 컬렉션 `exp_b2_qwen3emb06b_v1`은 **문서 5개(B2 청크 107개)만** 담은 실험용입니다. 전체 정책 237개가 들어 있지 않습니다.

**전제조건:** Docker에서 Qdrant 컨테이너 실행, `.env`의 `QDRANT_URL`·`QDRANT_API_KEY`. 꺼져 있으면 안내만 출력합니다.

In [ ]:
import warnings
from database import hybrid_search as hs

# 로컬(127.0.0.1) 개발 서버라 http로 접속한다 → qdrant-client의 "insecure connection" 경고만 숨긴다
warnings.filterwarnings("ignore", message="Api key is used with an insecure connection.")

def connect_qdrant():
    """실제 Qdrant에 접속해 본다. 실패하면 None (비밀값은 출력하지 않는다)."""
    if not RUN_DB_READ:
        print("RUN_DB_READ=False → Qdrant 조회를 건너뜁니다.")
        return None
    try:
        client = hs.make_qdrant_client()
        client.get_collections()          # 읽기 요청 1번으로 접속 확인
        return client
    except Exception as error:
        kind = getattr(error, "kind", type(error).__name__)
        print(f"Qdrant에 접속하지 못했습니다 ({kind}). Docker Desktop 실행 후 `docker compose up -d qdrant`를 확인하세요.")
        return None

qdrant_client = connect_qdrant()
QDRANT_OK = qdrant_client is not None
print("Qdrant 조회 가능:", QDRANT_OK)

**이번 셀의 목표:** 컬렉션 설정과 point 수를 읽습니다.
- 사용하는 개념: count·필터
- 입력 데이터: 실험 컬렉션
- 핵심 함수·매개변수: `store.count_points(client, name, filters)`
- 예상 출력: 1024 / Cosine / 107, 문서별 14·7·3·10·73
- 왜 실행하나요?: 저장된 실험 데이터가 기대와 같은지 확인

In [ ]:
COLLECTION = hs.DEFAULT_COLLECTION
if QDRANT_OK:
    params = qdrant_client.get_collection(COLLECTION).config.params.vectors
    print("벡터 차원:", params.size, "| 거리:", params.distance)
    print("전체 point:", store.count_points(qdrant_client, COLLECTION))
    for document_id in ["DOC_NHUF_003", "DOC_BOK_001", "DOC_YC_003", "DOC_GOV_006", "DOC_LH_001"]:
        print(f"  {document_id}: {store.count_points(qdrant_client, COLLECTION, {'document_id': document_id})}")
else:
    print("건너뜀 (Qdrant 미접속)")

**이번 셀의 목표:** point 1개의 구조를 풀어 봅니다.
- 사용하는 개념: point = id + vector + payload
- 입력 데이터: DOC_YC_003의 point 1개
- 핵심 함수·매개변수: `client.scroll(..., with_vectors=True)` (읽기)
- 예상 출력: payload 필드 목록, 벡터 길이 1024
- 왜 실행하나요?: 실제로 무엇이 저장되어 있는지 이해

In [ ]:
if QDRANT_OK:
    records, _ = qdrant_client.scroll(COLLECTION, scroll_filter=store._to_filter({"document_id": "DOC_YC_003"}),
                                      limit=1, with_payload=True, with_vectors=True)
    point = records[0]
    print("id      :", point.id)
    print("vector  : 길이", len(point.vector), "| 앞 5개", [round(x, 4) for x in point.vector[:5]])
    for key in ["chunk_id", "policy_id", "document_id", "section_title", "char_start", "char_end",
                "chunking_version", "embedding_model", "embedding_input_rule", "index_run_id"]:
        print(f"payload.{key:22}: {point.payload.get(key)}")
    print("payload.text 앞부분 :", point.payload["text"][:80].replace("\n", " "))
else:
    print("건너뜀 (Qdrant 미접속)")

### Chapter 5 정리
- point = id(=uuid5(chunk_id)) + vector(1024) + payload(원문·위치·출처·버전)
- upsert는 같은 id를 덮어쓰므로 재적재해도 늘지 않는다.
- 우리는 LangChain 대신 QdrantClient를 직접 써서 payload 구조·ID·안전장치를 직접 제어한다.
- 현재 컬렉션은 정책 5개 실험용이다.

**문제 5-1 (예상하기):** 메모리 모드 실습에서 `demo_points`를 세 번째로 upsert하면 point 수는?
**문제 5-2 (내 말로 설명하기):** 강의의 필터 키 `metadata.category`와 우리 필터 키 `document_id`가 다른 이유는?

---
# Chapter 6. Qdrant 검색

검색 흐름: 질문 → **질문 임베딩(query embedding)** → Qdrant에서 코사인이 높은 **Top-K** 청크 → 각 청크의 `policy_id`·`document_id`
- `score`: 질문 벡터와 청크 벡터의 코사인 유사도. **자격 충족 여부가 아닙니다.**
- metadata filter: `{"document_id": "DOC_LH_001"}`처럼 검색 범위를 좁힘

강의의 `similarity_search_with_score(query, k)`와 같은 역할을 우리 코드에서는 `hs.search_policy_evidence(query, filters, k)`가 합니다.

**이번 셀의 목표:** 실제 질문 1개로 Top-5를 검색합니다 (Ollama + Qdrant, 읽기 전용).
- 사용하는 개념: query embedding, Top-K, score
- 입력 데이터: 질문 문자열
- 핵심 함수·매개변수: `hs.search_policy_evidence(query, filters=None, k=5, client=qdrant_client)`
- 예상 출력: 1위가 DOC_YC_003 청크일 것으로 기대 (실험 기록 기준)
- 왜 실행하나요?: 강의의 유사도 검색을 실제 정책 데이터로 해 보기

In [ ]:
def show_hits(hits):
    print_table([{"rank": h["rank"], "score": round(h["score"], 4), "policy_id": h["policy_id"],
                  "document_id": h["document_id"], "chunk_id": h["chunk_id"], "page": h.get("page_start"),
                  "text": h["text"]} for h in hits],
                ["rank", "score", "policy_id", "document_id", "chunk_id", "page", "text"], width=36)

if OLLAMA_OK and QDRANT_OK:
    result = hs.search_policy_evidence("콘텐츠 기업이 인턴을 채용하면 1인당 인건비를 얼마 지원받나요?", k=5, client=qdrant_client)
    show_hits(result["hits"])
    print("시간(초):", result["latency_seconds"])
else:
    print("건너뜀: RUN_OLLAMA=True와 Qdrant 접속이 모두 필요합니다.")

## 6-1. 검색이 실패하는 사례: P3, P6 (저장된 실험 결과로 보기)

서버가 꺼져 있어도 볼 수 있도록 **노트북(11-05)에서 실제로 실행해 저장한 결과**(`docs/development/experiments/exp2_report.json`)를 읽습니다.
- **P3** "예비신혼부부는 언제까지 혼인신고를 해야 하나요?" — 근거 문장이 741자 청크 안 목록의 한 줄이라 의미가 희석된 것으로 **추정**
- **P6** "신청 관련 상담 전화번호는 무엇인가요?" — 정책 범위가 없는 일반 질문이라 다른 정책 청크가 상위를 차지

In [ ]:
saved = json.loads((ROOT / "docs/development/experiments/exp2_report.json").read_text(encoding="utf-8"))
for item in saved["search"]:
    if item["id"] in ("P3", "P6") and item["template"] == "A_none":
        print(f"[{item['id']}] {item['question']} | 근거 청크 {item['evidence_chunks']} | Top-5 안 근거 순위 {item['evidence_ranks']}")
        print_table([{"rank": h["rank"], "score": h["score"], "document_id": h["document_id"], "chunk_id": h["chunk_id"],
                      "근거?": "Y" if h["contains_evidence"] else "-"} for h in item["hits"]],
                    ["rank", "score", "document_id", "chunk_id", "근거?"])
        print()

**필터를 쓰면?** 데스크톱 실험(`desktop_20261105/hybrid_experiment.json`)에서 P6를 필터 없이 / `document_id` / `policy_id` 필터로 검색한 기록을 봅니다.

In [ ]:
hybrid_saved = json.loads((ROOT / "docs/development/experiments/desktop_20261105/hybrid_experiment.json").read_text(encoding="utf-8"))
print_table([{"필터": item["filter"], "지시문": item["template"], "근거 첫 순위": item["first_rank_containing_evidence"],
              "LH가 Top-5에": item["lh_in_top5"]} for item in hybrid_saved["p6"]],
            ["필터", "지시문", "근거 첫 순위", "LH가 Top-5에"])

**이번 셀의 목표:** (선택) 같은 질문을 실제로 필터 없이/있게 검색해 비교합니다.
- 사용하는 개념: metadata filter
- 입력 데이터: P6 질문
- 핵심 함수·매개변수: `hs.search_policy_evidence(query, filters={'document_id': 'DOC_LH_001'}, ...)`
- 예상 출력: 필터를 쓰면 DOC_LH_001 청크만 나옴
- 왜 실행하나요?: 필터가 검색 범위를 어떻게 바꾸는지 직접 확인

In [ ]:
if OLLAMA_OK and QDRANT_OK:
    p6 = "신청 관련 상담 전화번호는 무엇인가요?"
    for filters in (None, {"document_id": "DOC_LH_001"}):
        found = hs.search_policy_evidence(p6, filters=filters, k=5, client=qdrant_client)
        print("필터:", filters, "→ 문서:", [h["document_id"] for h in found["hits"]])
else:
    print("건너뜀: RUN_OLLAMA=True와 Qdrant 접속이 모두 필요합니다.")

### Chapter 6 정리
- 검색 = 질문 벡터와 코사인이 높은 청크 Top-K. score는 자격 판정이 아니다.
- 실패 사례: 긴 청크 안의 세부 내용(P3), 범위 없는 일반 질문(P6)
- 필터로 정책·문서 범위를 좁히면 P6의 근거 순위가 올라갔다(데스크톱 기록). 질문 재작성·필터 자동화는 아직 구현하지 않았다.

**문제 6-1 (예상하기):** `filters={"document_id": "DOC_LH_001"}`로 검색하면 결과에 `DOC_YC_003`이 나올 수 있을까요?
**문제 6-2 (내 말로 설명하기):** P3 실패의 원인 가설과 개선 아이디어 하나를 말해 보세요.

---
# Chapter 7. Neo4j 기초

## 7-1. Cypher 기본 문법

| 문법 | 뜻 | 예시 |
|---|---|---|
| `MATCH` | 패턴을 찾는다 | `MATCH (p:Policy) RETURN p.policy_name` |
| `WHERE` | 조건 | `MATCH (p:Policy) WHERE p.category = '주거·금융' RETURN p` |
| `RETURN` | 돌려줄 값 | `RETURN count(p) AS count` |
| `MERGE` | 있으면 찾고, 없으면 만든다 (쓰기) | `MERGE (p:Policy {policy_id: $id})` |
| 관계 방향 | `->`는 출발 → 도착 | `(p:Policy)-[:ADMINISTERED_BY]->(o:Organization)` |
| 파라미터 | 값은 `$이름`으로 따로 넘긴다 | `MATCH (p:Policy {policy_id: $policy_id})` + `{"policy_id": "POL_YC_003"}` |

**파라미터를 쓰는 이유:** 사용자 입력을 쿼리 문자열에 이어 붙이면, 입력에 Cypher 문법이 섞여 의도하지 않은 쿼리가 실행될 수 있습니다(주입).
값은 항상 `$파라미터`로 넘깁니다(AGENTS §4).

**이번 셀의 목표:** 가상 그래프에서 레이블 + ID로 노드를 찾는 MATCH를 흉내 냅니다.
- 사용하는 개념: 레이블로 범위 좁히기
- 입력 데이터: Chapter 1의 toy_nodes
- 핵심 함수·매개변수: 직접 만든 `toy_match(label, **props)`
- 예상 출력: 레이블이 맞는 노드만
- 왜 실행하나요?: Cypher MATCH의 의미를 파이썬으로 이해

In [ ]:
def toy_match(label=None, **properties):
    """Cypher의 MATCH (n:label {key: value})를 흉내 낸다. label=None이면 레이블을 보지 않는다."""
    return [node for node in toy_nodes
            if (label is None or node["label"] == label)
            and all(node.get(key) == value for key, value in properties.items())]

print(toy_match("Policy"))                        # MATCH (n:Policy)
print(toy_match("Organization", id="AGENCY_S2"))   # MATCH (n:Organization {id: 'AGENCY_S2'})

**이번 셀의 목표:** MERGE 쿼리가 어떻게 생겼는지 봅니다 (실행하지 않음).
- 사용하는 개념: MERGE + 파라미터
- 입력 데이터: 레이블 이름
- 핵심 함수·매개변수: `lg.build_node_merge_query('Policy')`
- 예상 출력: `UNWIND $rows AS row MERGE (n:Policy {policy_id: row.id}) …`
- 왜 실행하나요?: 우리 로더가 쓰는 실제 쿼리 모양을 보기 위해 — **DB에 실행하지 않는다**

In [ ]:
from database.neo4j import load_graph as lg

print(lg.build_node_merge_query("Policy"))
# 값(정책 ID 등)은 쿼리 문자열이 아니라 $rows 파라미터로 들어간다.

## 7-2. 실제 Neo4j 읽기 (읽기 전용)

**전제조건:** Docker의 Neo4j 실행, `.env`의 `NEO4J_URI`·`NEO4J_USER`·`NEO4J_PASSWORD`, **데이터가 적재된 DB**
(학원 노트북 Neo4j는 비어 있을 수 있습니다. 비어 있으면 0이 나오는 것이 정상입니다.)

아래 `neo4j_read`는 `hs.open_neo4j_runner()`로 **읽기 전용 실행기**를 열고, 쿼리 1개를 실행한 뒤 닫습니다.
이 실행기는 쓰기 메서드를 호출하면 `PermissionError`를 냅니다.

In [ ]:
def connect_neo4j_check():
    """Neo4j 접속을 한 번 확인한다. 실패하면 False (비밀번호는 출력하지 않는다)."""
    if not RUN_DB_READ:
        print("RUN_DB_READ=False → Neo4j 조회를 건너뜁니다.")
        return False
    try:
        with hs.open_neo4j_runner() as runner:
            runner.read_rows("RETURN 1 AS ok")
        return True
    except hs.HybridSearchBackendError as error:
        print(f"Neo4j에 접속하지 못했습니다 ({error.kind}). Docker Desktop 실행 후 `docker compose up -d neo4j`를 확인하세요.")
        return False

def neo4j_read(query, parameters=None):
    """읽기 전용 실행기로 Cypher 1개를 실행해 행 목록을 돌려준다. 접속 불가면 None."""
    if not NEO4J_OK:
        print("건너뜀 (Neo4j 미접속)")
        return None
    with hs.open_neo4j_runner() as runner:
        return runner.read_rows(query, parameters)

NEO4J_OK = connect_neo4j_check()
print("Neo4j 조회 가능:", NEO4J_OK)

**이번 셀의 목표:** 레이블별 노드 수를 읽습니다.
- 사용하는 개념: MATCH + RETURN + count
- 입력 데이터: Neo4j DB
- 핵심 함수·매개변수: `neo4j_read("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count")`
- 예상 출력: 데이터가 있으면 Policy 237 등 9종, 비어 있으면 빈 결과
- 왜 실행하나요?: DB 상태를 직접 확인

In [ ]:
rows = neo4j_read("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY count DESC")
if rows is not None:
    print_table(rows, ["label", "count"]) if rows else print("노드가 없습니다 (빈 DB).")

**이번 셀의 목표:** 파라미터를 써서 정책 1개를 찾습니다.
- 사용하는 개념: MATCH + 파라미터
- 입력 데이터: `{'policy_id': 'POL_YC_003'}`
- 핵심 함수·매개변수: `neo4j_read(query, parameters)`
- 예상 출력: 정책명 1줄
- 왜 실행하나요?: 값을 문자열에 이어 붙이지 않고 파라미터로 넘기는 방법

In [ ]:
query = "MATCH (p:Policy {policy_id: $policy_id}) RETURN p.policy_name AS name, p.category AS category"
rows = neo4j_read(query, {"policy_id": "POL_YC_003"})
if rows is not None:
    print(rows if rows else "결과 없음 (빈 DB이거나 정책이 없음)")

### Chapter 7 정리
- MATCH(찾기)·WHERE(조건)·RETURN(결과)·MERGE(있으면 찾고 없으면 생성)
- 관계에는 방향(`->`)이 있다.
- 값은 `$파라미터`로 넘긴다(주입 방지).
- 이 노트북은 Neo4j에 쓰지 않는다. MERGE는 모양만 본다.

**문제 7-1 (예상하기):** `toy_match(None, id="POL_S1")`과 `toy_match("Organization", id="POL_S1")`의 결과는?
**문제 7-2 (빈칸 채우기):** 아래 `...`을 채워 Organization 노드만 찾으세요.

In [ ]:
# 문제 7-2: ... 을 알맞은 레이블 문자열로 바꾸세요
label_name = ...
if label_name is ...:
    print("빈칸을 채워 보세요")
else:
    print(toy_match(label_name))

---
# Chapter 8. Neo4j 적재 과정 이해 (`database/neo4j/load_graph.py`)

| 단계 | 함수 | 하는 일 |
|---|---|---|
| ① 읽기 | `read_graph_input()` | 인계 노드 9종 + 관계 파일 읽기 |
| ② 검증 | `validate_graph_input(graph)` | ID 누락·중복, 저장 불가 속성, 허용되지 않은 레이블·관계, 양끝 노드 없음, 중복 관계 → 하나라도 있으면 **쓰기 전에 중단** |
| ③ 스키마 | `build_schema_statements()` | 고유 제약조건 9개 + 인덱스 4개 (`IF NOT EXISTS`) |
| ④ 계획 | `build_load_plan(graph)` | 노드 → 관계 순서, 500행 배치 |
| ⑤ 적재 전 보호 | `preflight_database(graph, runner)` | 비어 있지 않은 DB에는 기본적으로 거부 (`allow_existing=True`를 명시해야 진행) |
| ⑥ 적재 | `load_graph(graph, runner)` | 배치마다 트랜잭션 1개. MERGE 건수가 다르면 그 배치만 롤백 |
| ⑦ 사후 검증 | `verify_loaded_graph(graph, runner)` | 건수·스키마·노드 ID·관계 양끝 ID·속성값 원본 대조 |

이 장에서는 ①~④를 **오프라인**으로 실행하고, 실제 DB는 읽기 전용 검증만 합니다. (적재 ⑤⑥은 실행하지 않음)

**이번 셀의 목표:** 인계 데이터를 검증합니다 (DB 없이).
- 사용하는 개념: 쓰기 전 검증
- 입력 데이터: `data/handoff/neo4j/`
- 핵심 함수·매개변수: `lg.read_graph_input()`, `lg.validate_graph_input(graph)`
- 예상 출력: `ok: True`, 노드 1,034 / 관계 3,615
- 왜 실행하나요?: 적재 전에 입력에 문제가 없는지 확인

In [ ]:
graph = lg.read_graph_input()
report = lg.validate_graph_input(graph)
print("검증 통과:", report.ok)
print("노드 합계:", sum(report.node_counts.values()), report.node_counts)
print("관계 합계:", sum(report.relationship_counts.values()))

**이번 셀의 목표:** 스키마 문장(제약조건 9 + 인덱스 4)을 봅니다.
- 사용하는 개념: 고유 제약조건·인덱스
- 입력 데이터: 허용 목록
- 핵심 함수·매개변수: `lg.build_schema_statements()`
- 예상 출력: 13개 문장
- 왜 실행하나요?: MERGE가 같은 ID 노드를 하나만 만들도록 DB 차원에서 보장하는 장치를 이해

In [ ]:
statements = lg.build_schema_statements()
print("문장 수:", len(statements))
for statement in statements:
    print(" ", statement)

**이번 셀의 목표:** 관계 MERGE 쿼리를 봅니다.
- 사용하는 개념: 레이블 + 고유 ID로 양끝 찾기
- 입력 데이터: 관계 타입 이름
- 핵심 함수·매개변수: `lg.build_relationship_merge_query('ADMINISTERED_BY')`
- 예상 출력: `MATCH (s:Policy {policy_id: …})` 형태
- 왜 실행하나요?: 관계를 만들 때 출발·도착 레이블을 명시한다는 것을 확인

In [ ]:
print(lg.build_relationship_merge_query("ADMINISTERED_BY"))
plan = lg.build_load_plan(graph)
print("\n쓰기 단계(배치) 수:", len(plan), "| 노드 단계:", sum(s.kind == "node" for s in plan),
      "| 관계 단계:", sum(s.kind == "relationship" for s in plan))

## 8-1. 인계 로더에서 관계가 11,316개로 늘었던 이유

인계 로더(`data/handoff/neo4j/load_neo4j.py`, **실행하지 않는 파일**)는 관계를 만들 때 이렇게 찾았습니다.

```cypher
MATCH (src) WHERE src.policy_id = row.source_id OR src.document_id = row.source_id OR ...
```

레이블이 없어서 "**그 속성을 가진 모든 노드**"가 걸립니다. 그런데 `Document`와 `ConditionGroup` 노드에도 `policy_id` 속성이 있습니다.

| | 잘못된 MATCH (인계 로더) | 올바른 MATCH (새 로더) |
|---|---|---|
| 패턴 | `MATCH (src) WHERE src.policy_id = $id OR …` | `MATCH (s:Policy {policy_id: $id})` |
| `POL_001`이 걸리는 노드 | Policy 1 + Document 3 + ConditionGroup 1 = **5개** | Policy **1개** |
| 관계 1행이 만드는 관계 | 출발 후보 × 도착 후보 개수만큼 | 정확히 1개 |
| 전체 관계 | **11,316개** (오프라인 재현) | **3,615개** |

**이번 셀의 목표:** 작은 가상 노드로 결함을 재현합니다.
- 사용하는 개념: 레이블 없는 매칭의 문제
- 입력 데이터: 가상 노드 5개 (sample_only)
- 핵심 함수·매개변수: 직접 만든 `legacy_match`, `labeled_match`
- 예상 출력: 잘못된 방식 4개 / 올바른 방식 1개
- 왜 실행하나요?: 왜 관계가 부풀었는지 눈으로 확인

In [ ]:
# 가상 노드 (sample_only): 정책 1, 그 정책의 문서 2, 조건 그룹 1, 기관 1
virtual_nodes = [
    {"label": "Policy", "policy_id": "POL_V"},
    {"label": "Document", "document_id": "DOC_V1", "policy_id": "POL_V"},
    {"label": "Document", "document_id": "DOC_V2", "policy_id": "POL_V"},
    {"label": "ConditionGroup", "group_id": "GRP_V", "policy_id": "POL_V"},
    {"label": "Organization", "agency_id": "AGENCY_V"},
]
relationship_row = {"source_id": "POL_V", "relation": "ADMINISTERED_BY", "target_id": "AGENCY_V"}

def legacy_match(value):
    """인계 로더 방식: 레이블 없이 여러 ID 속성 중 하나라도 같으면 매칭"""
    return [n for n in virtual_nodes if value in (n.get("policy_id"), n.get("document_id"), n.get("group_id"), n.get("agency_id"))]

def labeled_match(label, id_field, value):
    """새 로더 방식: 레이블 + 그 레이블의 고유 ID 필드로만 매칭"""
    return [n for n in virtual_nodes if n["label"] == label and n.get(id_field) == value]

legacy_pairs = len(legacy_match(relationship_row["source_id"])) * len(legacy_match(relationship_row["target_id"]))
new_pairs = len(labeled_match("Policy", "policy_id", "POL_V")) * len(labeled_match("Organization", "agency_id", "AGENCY_V"))
print("잘못된 방식이 만드는 관계 수:", legacy_pairs, "| 걸린 출발 노드:", [n["label"] for n in legacy_match("POL_V")])
print("올바른 방식이 만드는 관계 수:", new_pairs)

**이번 셀의 목표:** 실제 인계 데이터 전체로 재현합니다 (DB 없이).
- 사용하는 개념: 결함 재현
- 입력 데이터: 인계 노드·관계 전체
- 핵심 함수·매개변수: 직접 만든 `simulate_legacy(graph)`
- 예상 출력: 11316
- 왜 실행하나요?: 문서에 적힌 숫자를 직접 확인

In [ ]:
from collections import defaultdict

def simulate_legacy(graph):
    """인계 로더의 OR 매칭 규칙을 그대로 흉내 내 만들어질 관계 수를 센다. (테스트 코드와 같은 규칙)"""
    source_fields = ("policy_id", "document_id", "group_id", "agency_id", "region_id")
    target_fields = source_fields + ("category_id", "stage_id", "condition_id")
    all_nodes = [node for rows in graph.nodes_by_label.values() for node in rows]
    by_source, by_target = defaultdict(set), defaultdict(set)
    for index, node in enumerate(all_nodes):
        for field in source_fields:
            if isinstance(node.get(field), str):
                by_source[node[field]].add(index)
        for field in target_fields:
            if isinstance(node.get(field), str):
                by_target[node[field]].add(index)
    created = set()
    for row in graph.relationships:
        for s in by_source[row["source_id"]]:
            for t in by_target[row["target_id"]]:
                created.add((row["relation"], s, t))
    return len(created)

print("인계 로더 방식:", simulate_legacy(graph), "| 새 로더 방식:", len(graph.relationships))

## 8-2. 트랜잭션과 재적재 보호
- **트랜잭션:** 배치(최대 500행) 1개 = 쓰기 트랜잭션 1개. 배치 안 MERGE 건수가 행 수와 다르면 그 배치만 롤백됩니다. 앞서 커밋된 배치는 남지만, 모두 MERGE라 같은 입력으로 다시 실행하면 중복 없이 완성됩니다.
- **재적재 보호(`preflight_database`):** 비어 있지 않은 DB에는 기본적으로 적재를 거부합니다(`DatabaseNotEmptyError`). `allow_existing=True`를 명시하면, 기존 데이터가 입력의 부분집합인지 먼저 확인한 뒤 진행합니다.
- 데스크톱 실측: 노드 1,034 / 관계 3,615, 재실행 중복 0, 속성값 원본 대조 모두 일치.

**이번 셀의 목표:** (DB가 적재되어 있으면) 사후 검증 함수를 읽기 전용으로 실행합니다.
- 사용하는 개념: 적재 후 검증
- 입력 데이터: Neo4j DB + 인계 데이터
- 핵심 함수·매개변수: `lg.snapshot_counts(runner)`, `lg.verify_loaded_graph(graph, runner)`
- 예상 출력: 적재된 DB면 `ok: True`, 빈 DB면 건수 불일치로 `ok: False`
- 왜 실행하나요?: DB가 인계 파일과 같은지 직접 확인

In [ ]:
if NEO4J_OK:
    with hs.open_neo4j_runner() as runner:          # 읽기 전용 실행기
        counts = lg.snapshot_counts(runner)
        print("노드 합계:", sum(v for k, v in counts.items() if k.startswith("node:")),
              "| 관계 합계:", sum(v for k, v in counts.items() if k.startswith("rel:")))
        if sum(counts.values()):
            verification = lg.verify_loaded_graph(graph, runner)
            print("사후 검증 ok:", verification["ok"])
        else:
            print("빈 DB입니다 → 사후 검증 생략 (적재는 이 노트북에서 하지 않습니다)")
else:
    print("건너뜀 (Neo4j 미접속)")

### Chapter 8 정리
- 검증 → 스키마 → 노드 → 관계 → 사후 검증 순서
- 레이블 + 고유 ID로만 MATCH/MERGE 한다 → 11,316이 아니라 3,615
- 배치 트랜잭션 + 재적재 보호 + 사후 검증

**문제 8-1 (예상하기):** 가상 예제에서 정책에 문서가 하나 더 있으면(문서 3개) 잘못된 방식의 관계 수는?
**문제 8-2 (내 말로 설명하기):** `MERGE`만 쓰면 중복이 안 생기는데, 고유 제약조건은 왜 따로 만드나요?

---
# Chapter 9. GraphDB에서 정책 상세정보 조회

`hs.get_policy_graph_context(policy_ids, runner=...)`는 정책 ID 목록을 받아 **읽기 전용 Cypher 2개**로 다음을 돌려줍니다.
- `policy`(정책 속성), `organizations`, `regions`, `categories`, `target_groups`, `life_stages`, `documents`
- `eligibility.condition_tree`: 루트 `ConditionGroup`부터 하위 그룹·`EligibilityCondition`까지의 트리
- `eligibility.evaluated`는 항상 `False` — **자격을 판정하지 않습니다.**

## AND / OR / NOT / UNKNOWN
| logic | 뜻 |
|---|---|
| `AND` | 그룹 안 조건을 모두 만족해야 함 |
| `OR` | 하나만 만족하면 됨 |
| `NOT` | 해당하면 안 됨 |
| `UNKNOWN` | 구조화된 조건이 없음 → **"자격 제한 없음"이 아니라 "공고 원문 확인 필요"** |

**이번 셀의 목표:** 실제 Neo4j에서 정책 1개(POL_GOV_006)의 그래프 문맥을 읽습니다.
- 사용하는 개념: 정책 상세 조회
- 입력 데이터: `['POL_GOV_006']`
- 핵심 함수·매개변수: `hs.get_policy_graph_context(policy_ids, runner=runner)`
- 예상 출력: 정책명, 기관, 지역, 문서, 조건 트리
- 왜 실행하나요?: Neo4j가 정책 관계를 어떻게 돌려주는지 확인

In [ ]:
graph_context = None
if NEO4J_OK:
    with hs.open_neo4j_runner() as runner:
        graph_context = hs.get_policy_graph_context(["POL_GOV_006"], runner=runner)
    item = graph_context["policies"].get("POL_GOV_006")
    if item:
        print("정책:", item["policy"]["policy_name"], "| verification_status:", item["policy"].get("verification_status"))
        print("기관:", [o.get("name") for o in item["organizations"]])
        print("지역:", [r.get("name") for r in item["regions"]])
        print("문서:", [d.get("document_id") for d in item["documents"]])
        print("관계 속성 예시:", item["organizations"][0]["relationship"] if item["organizations"] else None)
    else:
        print("정책이 그래프에 없습니다 (빈 DB?) → missing:", graph_context["missing_policy_ids"])
else:
    print("건너뜀 (Neo4j 미접속) — 아래 오프라인 셀에서 같은 정책의 조건 구조를 인계 파일로 볼 수 있습니다.")

**이번 셀의 목표:** 조건 트리를 들여쓰기로 출력합니다.
- 사용하는 개념: 재귀 함수
- 입력 데이터: condition_tree
- 핵심 함수·매개변수: 직접 만든 `print_tree(group)`
- 예상 출력: ROOT(AND) 아래 OR 그룹 2개
- 왜 실행하나요?: AND/OR 계층 구조를 눈으로 보기

In [ ]:
def print_tree(group, depth=0):
    """ConditionGroup 트리를 들여쓰기로 출력한다 (판정하지 않고 구조만 보여 준다)."""
    pad = "    " * depth
    print(f"{pad}[{group.get('logic')}] {group.get('group_id')}")
    for condition in group.get("conditions", []):
        print(f"{pad}    - {condition.get('condition_id')} ({condition.get('name')})")
    for child in group.get("groups", []):
        print_tree(child, depth + 1)

if graph_context and graph_context["policies"].get("POL_GOV_006"):
    for tree in graph_context["policies"]["POL_GOV_006"]["eligibility"]["condition_tree"]:
        print_tree(tree)
else:
    print("건너뜀 (그래프 문맥 없음)")

**이번 셀의 목표:** (오프라인) 인계 파일로 같은 조건 구조를 만들어 봅니다.
- 사용하는 개념: 그래프 데이터를 파이썬으로 따라가기
- 입력 데이터: 인계 ConditionGroup 노드·관계
- 핵심 함수·매개변수: dict·리스트 필터
- 예상 출력: ROOT(AND): 연령·무주택 + OR 그룹 2개
- 왜 실행하나요?: DB 없이도 조건 구조를 확인

In [ ]:
groups = {g["group_id"]: g for g in nodes_by_label["ConditionGroup"]}

def offline_tree(group_id, depth=0):
    pad = "    " * depth
    print(f"{pad}[{groups[group_id]['logic']}] {group_id}")
    for rel in handoff_relationships:
        if rel["source_id"] == group_id and rel["relation"] == "CONTAINS_CONDITION":
            print(f"{pad}    - {rel['target_id']}")
    for rel in handoff_relationships:
        if rel["source_id"] == group_id and rel["relation"] == "CONTAINS_GROUP":
            offline_tree(rel["target_id"], depth + 1)

offline_tree("GRP_POL_GOV_006_ROOT")

**이번 셀의 목표:** 조건 노드를 여러 정책이 공유하는지 셉니다.
- 사용하는 개념: 노드 재사용
- 입력 데이터: REQUIRES_CONDITION 관계
- 핵심 함수·매개변수: `Counter`
- 예상 출력: 가장 많이 쓰인 조건 5개
- 왜 실행하나요?: 조건을 정책마다 복사하지 않고 노드 하나를 공유하는 이유 이해

In [ ]:
shared = Counter(r["target_id"] for r in handoff_relationships if r["relation"] == "REQUIRES_CONDITION")
print("서로 다른 조건 노드 수:", len(shared))
print_table([{"condition_id": cid, "연결된 정책 수": n} for cid, n in shared.most_common(5)], ["condition_id", "연결된 정책 수"])

**조건 노드를 공유하는 이유:** "중위소득 50% 이하" 같은 같은 조건을 정책마다 따로 만들면 이름·값이 조금씩 달라질 수 있습니다.
노드 하나를 공유하면 "이 조건을 쓰는 정책 모두"를 한 번에 찾을 수 있습니다.

**관계 속성:** `evidence_document_id`(근거 문서), `evidence_location`(근거 위치), `review_status`(`verified` / `extracted`), `target_name`.
`extracted`는 자동 추출값이며 공식 검증 완료가 아닙니다.

> 자격 충족 여부 판정은 **강민지님 담당(backend/eligibility/)**입니다. 이 장의 코드는 구조만 보여 줍니다.

### Chapter 9 정리
- 정책 → 기관·지역·문서·조건 그룹 관계를 Neo4j가 돌려준다.
- 조건 트리는 AND/OR/NOT/UNKNOWN 계층을 그대로 보존한다(해석·판정하지 않음).
- UNKNOWN = 구조화 조건 없음 ≠ 제한 없음
- 조건 노드는 여러 정책이 공유한다.

**문제 9-1 (예상하기):** 루트가 `UNKNOWN`이고 조건이 0개인 정책은 "누구나 받을 수 있다"는 뜻일까요?
**문제 9-2 (내 말로 설명하기):** `review_status=extracted`인 관계를 답변 근거로 그대로 쓰면 왜 위험한가요?

---
# Chapter 10. Qdrant + Neo4j 연결 (이 프로젝트의 핵심)

`search_policy_context(query, filters=None, k=5)`가 하는 일을 작은 단계로 나눠 봅니다.

```
1. 질문 입력 → 입력 검증 (validate_query / validate_k / validate_filters)
2. Ollama로 질문 임베딩
3. Qdrant 검색 (search_policy_evidence)          → hits: 원문 근거 청크 + policy_id
4. policy_id 추출 (검색 순서 유지, 형식 검사)
5. Neo4j에서 정책 조회 (get_policy_graph_context) → 기관·지역·조건 트리
6. 두 결과 결합 (combine_policy_evidence)         → 정책별 {evidence, graph, data_quality}
7. 반환: policies, latency_seconds, notes …
```

**이번 셀의 목표:** 입력 검증이 DB 접속 전에 잘못된 입력을 막는 것을 봅니다 (DB 없이).
- 사용하는 개념: 입력 검증
- 입력 데이터: 잘못된 k, 허용되지 않은 필터
- 핵심 함수·매개변수: `hs.validate_k`, `hs.validate_filters`, `hs.validate_query`
- 예상 출력: `HybridSearchInputError` 메시지 3개
- 왜 실행하나요?: 서버에 가기 전에 오류를 잡는 이유 이해

In [ ]:
for check, value in [(hs.validate_k, 0), (hs.validate_filters, {"text": "x"}), (hs.validate_query, "   ")]:
    try:
        check(value)
        print("통과:", value)
    except hs.HybridSearchInputError as error:
        print(f"{check.__name__}({value!r}) → 거부: {error}")
print("허용 필터 필드:", hs.ALLOWED_FILTER_FIELDS)

**이번 셀의 목표:** 단계 3~6을 하나씩 직접 호출합니다 (Ollama + Qdrant + Neo4j).
- 사용하는 개념: 단계별 실행
- 입력 데이터: 질문 1개
- 핵심 함수·매개변수: `hs.search_policy_evidence` → policy_id 목록 → `hs.get_policy_graph_context` → `hs.combine_policy_evidence`
- 예상 출력: 정책별 근거 수와 graph_status
- 왜 실행하나요?: search_policy_context 내부를 이해

In [ ]:
question = "신혼부부 전세자금 대출의 금리는 어떻게 되나요?"
if OLLAMA_OK and QDRANT_OK and NEO4J_OK:
    search_result = hs.search_policy_evidence(question, k=5, client=qdrant_client)        # 3. Qdrant
    policy_ids = list(dict.fromkeys(h["policy_id"] for h in search_result["hits"]))         # 4. 순서 유지 + 중복 제거
    print("검색된 policy_id:", policy_ids)
    with hs.open_neo4j_runner() as runner:
        graph_part = hs.get_policy_graph_context(policy_ids, runner=runner)                # 5. Neo4j
    combined = hs.combine_policy_evidence(search_result, graph_part)                        # 6. 결합
    for policy in combined["policies"]:
        print(policy["policy_id"], "| 최고 순위", policy["best_rank"], "| 근거", len(policy["evidence"]),
              "개 | graph_status", policy["graph_status"])
else:
    print("건너뜀: RUN_OLLAMA=True, Qdrant·Neo4j 접속이 모두 필요합니다.")

**이번 셀의 목표:** 통합 함수 한 번 호출과 세션 재사용을 비교합니다.
- 사용하는 개념: 연결 재사용
- 입력 데이터: 같은 질문 3번
- 핵심 함수·매개변수: `hs.search_policy_context(...)`, `hs.HybridSearchSession()`
- 예상 출력: 세션 재사용이 더 빠름(실행 환경마다 수치 다름)
- 왜 실행하나요?: 질문을 여러 번 보낼 때 왜 세션을 쓰는지 이해

In [ ]:
import time
if OLLAMA_OK and QDRANT_OK and NEO4J_OK:
    start = time.perf_counter()
    for _ in range(3):
        hs.search_policy_context(question, k=5)            # 호출마다 연결을 열고 닫는다
    each_call = (time.perf_counter() - start) / 3

    start = time.perf_counter()
    with hs.HybridSearchSession() as session:               # 연결을 한 번 만들어 재사용, with가 끝나면 닫힌다
        for _ in range(3):
            session.search_policy_context(question, k=5)
    reused = (time.perf_counter() - start) / 3
    print(f"매번 호출: {each_call:.3f}초/회 | 세션 재사용: {reused:.3f}초/회 (이 PC 실측)")
else:
    print("건너뜀. 참고: 데스크톱 기록은 매번 호출 약 0.162초, 세션 재사용 약 0.026초 (인수인계서 §3-E)")

**결과 구조 (주요 키)**
- `policies[]`: `policy_id`, `policy_name`, `best_rank`, `best_score`, `evidence[]`(원문 청크·위치·출처), `graph_status`(`FOUND` / `MISSING_IN_GRAPH`), `graph`(기관·지역·조건 트리), `data_quality.warnings`
- `latency_seconds`: `query_embedding`, `qdrant_search`, `neo4j_context`
- `notes`: score·eligibility·UNKNOWN 해석 주의 문구

**원문 근거와 그래프 정보는 다릅니다:** 답변 근거·출처는 `evidence[].text`(공식 원문 청크)를 씁니다. `graph.policy`의 `eligibility`·`benefit`은 수집 단계 추출 텍스트입니다.

**오류 종류:** 입력 문제 → `HybridSearchInputError`, 서버 문제 → `HybridSearchBackendError`(`.backend`: ollama/qdrant/neo4j, `.kind`: unavailable/timeout/auth/config/server). 코드 버그는 감싸지 않고 원래 예외로 올립니다.

### Chapter 10 정리
- 벡터 검색으로 policy_id를 얻고, 그 ID로 그래프 문맥을 붙인다.
- 입력 검증은 서버 접속 전에 한다.
- 질문이 여러 개면 `HybridSearchSession`으로 연결을 재사용한다.

**문제 10-1 (예상하기):** Qdrant가 돌려준 policy_id가 Neo4j에 없으면 결과의 `graph_status`는?
**문제 10-2 (빈칸 채우기):** 아래 `...`을 채워 k=3으로 검증을 통과시키세요.

In [ ]:
# 문제 10-2: ... 을 1~200 사이 정수로 바꾸세요
k_value = ...
if k_value is ...:
    print("빈칸을 채워 보세요")
else:
    print("검증된 k:", hs.validate_k(k_value))

---
# Chapter 11. 강의 Hybrid RAG와 실제 구현 비교

## 11-1. 강의 `4. Hybrid RAG.ipynb`의 구성 (강의 코드 기준)

| 강의 요소 | 강의 코드에서 하는 일 |
|---|---|
| `HybridRAGState`(TypedDict) | `question`, `selected_node`, `vector_docs`, `tool_calls`, `graph_docs`, `reranked_docs`, `relevant_docs`, `answer`를 노드끼리 공유 |
| `supervisor_node` | LLM(`ChatOllama(model="gemma4:e2b")`)이 질문을 보고 `graphdb` 또는 `vectordb` 선택 |
| `vector_search_node` | **Elasticsearch**(`ElasticsearchVectorStore.hybrid_search_with_score`, BM25 + 벡터, 0~1 정규화 점수) k=3 |
| Graph Tool 3개 | `search_news_by_category` / `_by_publisher` / `_by_reporter` (`@tool`, Neo4j 쿼리 템플릿) |
| `select_graph_tool_node` | `ChatOpenAI(model="gpt-5-nano").bind_tools(tools)`가 도구 선택 (Tool Calling) |
| `execution_graph_tool_node` | 선택된 도구 실행 |
| `evaluate_with_llm_node` | LLM이 문서 관련성 점수를 매겨 3점 이상만 남김 |
| `rerank_with_score_node` | 점수 내림차순 정렬 (설명에는 "0.5 이상 필터"가 있지만 **코드는 정렬만** 함) |
| `answer_generator_node` | 관련 문서로 답변 생성 + 출처 추가 |
| 워크플로우 | `START → supervisor → (vector_search | select_graph_tool → execution_graph_tool → vector_search) → evaluate_with_llm → rerank_with_score → answer_generator → END` |

## 11-2. 비교

```
강의:        질문 분석(Supervisor) → 검색 경로 선택 → (그래프 도구 +) 벡터 검색 → LLM 평가 → 재순위화 → 답변
LifePolicy: 질문 임베딩 → Qdrant Top-K → policy_id → Neo4j 정책 문맥 → 결합 (답변은 다른 팀원 영역)
```

| 항목 | 강의 | LifePolicy (현재 코드) |
|---|---|---|
| LangGraph State·워크플로우 | 있음 | **없음** (일반 파이썬 함수·클래스) |
| Supervisor(검색 경로 선택) | 있음 | **없음** — 항상 벡터 검색 → 그래프 문맥 순서 |
| 벡터 저장소 | Elasticsearch | **Qdrant** |
| BM25(키워드 검색) | 있음 (Elasticsearch 하이브리드) | **없음** |
| 그래프 사용 방식 | 도구로 **독립 검색**(카테고리·언론사·기자) | 벡터 검색 결과의 policy_id로 **문맥 보강** |
| Tool Calling | 있음 (OpenAI) | **없음** |
| 재순위화·LLM 평가 | 있음 | **없음** |
| 답변 생성 | 있음 | **없음** (나지인·박소희 담당) |
| 비밀번호 | 도구 코드 안에 직접 작성 | `.env`에서 읽고 출력하지 않음 |
| 결과 형태 | `(Document, score)` 튜플 | JSON 직렬화 가능한 dict (출처·위치·버전 포함) |

**우리 코드에 Supervisor, BM25, LangGraph 워크플로우, LLM 평가가 있다고 설명하면 틀립니다.**

**이번 셀의 목표:** 우리 결과 구조의 최상위 키를 강의 State 필드와 나란히 봅니다 (저장된 결과 사용).
- 사용하는 개념: 결과 계약
- 입력 데이터: 데스크톱 실험 결과 JSON
- 핵심 함수·매개변수: dict.keys()
- 예상 출력: policies, latency_seconds 등
- 왜 실행하나요?: RAG 담당이 받을 데이터 모양을 이해

In [ ]:
lecture_state_fields = ["question", "selected_node", "vector_docs", "tool_calls", "graph_docs",
                        "reranked_docs", "relevant_docs", "answer"]
our_policy_fields = sorted(hybrid_saved["questions"][0]["policies"][0].keys())
print("강의 HybridRAGState:", lecture_state_fields)
print("우리 policies[] 항목:", our_policy_fields)

## 11-3. 앞으로 적용할 수 있는 확장 (현재 미구현)
- **BM25 + 벡터 결합:** 전화번호·정책번호처럼 정확한 낱말이 중요한 질문(P6 유형)에 도움이 될 수 있음
- **Supervisor/질문 재작성:** 질문에서 정책·지역을 뽑아 Qdrant 필터(`policy_id`, `region`)를 자동으로 거는 방식
- **재순위화(rerank):** Top-K 후보를 더 정밀한 모델로 다시 정렬
- **LangGraph:** 검색 → 조건 확인 → 답변 → 추가 질문을 상태 기반 워크플로우로 연결 (RAG 담당과 협의)

### Chapter 11 정리
- 강의는 "어떤 검색을 쓸지 고르는" 에이전트형, 우리는 "벡터 검색 + 그래프 문맥 보강"형
- BM25·Supervisor·LangGraph·LLM 평가는 아직 없다(확장 후보).

**문제 11-1 (내 말로 설명하기):** 강의의 그래프 검색과 우리 그래프 사용 방식의 차이는?
**문제 11-2 (예상하기):** P6 같은 "전화번호" 질문에 BM25가 도움이 될 수 있는 이유는?

---
# Chapter 12. 실제 서비스 연결 (가상 사용자 예시)

| 구분 | 내용 |
|---|---|
| 가상 사용자 프로필 (sample_only) | 29세, 서울 거주, 신혼부부, 무주택 |
| 질문 | "제가 받을 수 있는 신혼부부 주거 지원 정책이 있나요?" |
| 실제로 실행되는 부분 | ①~④ Hybrid 검색 (현재 컬렉션의 정책 5개 범위 안에서만) |
| **가상 흐름** | ⑤ 자격 확인(강민지 담당), ⑥ LLM 답변(나지인·박소희 담당) — 이 노트북은 구현하지 않고 "전달할 데이터"만 보여 줌 |

In [ ]:
virtual_profile = {"age": 29, "region": "서울특별시", "family": "신혼부부", "housing": "무주택", "note": "sample_only"}
service_question = "제가 받을 수 있는 신혼부부 주거 지원 정책이 있나요?"
print("가상 프로필:", virtual_profile)
print("질문:", service_question)

**이번 셀의 목표:** 실제 Hybrid 검색으로 관련 정책 후보를 찾습니다.
- 사용하는 개념: search_policy_context
- 입력 데이터: 질문
- 핵심 함수·매개변수: `hs.search_policy_context(query, k=5)`
- 예상 출력: 정책 후보, 근거 수, 데이터 품질 경고 코드
- 왜 실행하나요?: 실제 시스템에서 검색 결과가 어떤 모양으로 넘어가는지 확인

In [ ]:
service_result = None
if OLLAMA_OK and QDRANT_OK and NEO4J_OK:
    service_result = hs.search_policy_context(service_question, k=5)
    for policy in service_result["policies"]:
        warnings = [w["code"] for w in policy["data_quality"]["warnings"]]
        print(policy["policy_id"], policy["policy_name"][:30], "| 근거", len(policy["evidence"]), "| 경고", warnings)
else:
    print("건너뜀: RUN_OLLAMA=True, Qdrant·Neo4j 접속이 모두 필요합니다.")

**이번 셀의 목표:** (가상 흐름) 다음 담당자에게 넘길 데이터 모양만 만듭니다.
- 사용하는 개념: 인터페이스
- 입력 데이터: service_result
- 핵심 함수·매개변수: 직접 만든 dict
- 예상 출력: 근거 텍스트·출처 URL 목록
- 왜 실행하나요?: 자격 판정·답변 생성이 무엇을 받아야 하는지 이해 — **판정·답변은 하지 않음**

In [ ]:
if service_result:
    handoff_to_teammates = [
        {"policy_id": p["policy_id"],
         "evidence": [{"text": e["text"][:60], "source_url": e["source_url"], "page": e.get("page_start")} for e in p["evidence"][:2]],
         "eligibility_evaluated": p["graph"]["eligibility"]["evaluated"] if p.get("graph") else None}
        for p in service_result["policies"]]
    print(json.dumps(handoff_to_teammates, ensure_ascii=False, indent=2)[:1500])
print("⑤ 자격 확인: 가상 흐름 (강민지 담당, 이 노트북에서 판정하지 않음)")
print("⑥ LLM 답변: 가상 흐름 (나지인·박소희 담당, 이 노트북에서 LLM을 호출하지 않음)")

### Chapter 12 정리
- 실제로 동작하는 것은 검색·결합까지이고, 범위는 정책 5개다.
- 자격 확인·답변 생성은 다른 담당자 영역이다.
- `eligibility.evaluated=False`는 "아직 판정하지 않았다"는 뜻이다.

**문제 12-1 (내 말로 설명하기):** 가상 사용자가 "서울 거주"인데 검색 결과에 다른 지역 정책이 나올 수 있는 이유는?

---
# Chapter 13. 데이터 품질 문제

## 13-1. 두 가지 "정확성"
| 구분 | 뜻 | 현재 |
|---|---|---|
| 저장·관계 무결성 | DB가 인계 파일과 같은가 | 통과 (데스크톱 실측) |
| 의미 정확성 | 그래프 자격 조건이 실제 정책 자격과 맞는가 | **오류 의심 다수, 미검증** |

**DB가 인계 파일과 같다고 해서 정책 조건이 정확하다는 뜻은 아닙니다.**

**이번 셀의 목표:** UNKNOWN 정책 수를 셉니다 (인계 파일, DB 없이).
- 사용하는 개념: UNKNOWN
- 입력 데이터: ConditionGroup 노드
- 핵심 함수·매개변수: `Counter`
- 예상 출력: 루트 237개 중 UNKNOWN 155 / AND 82
- 왜 실행하나요?: 조건 구조가 없는 정책이 얼마나 많은지 확인

In [ ]:
roots = [g for g in nodes_by_label["ConditionGroup"] if g["is_root"]]
print("루트 그룹:", len(roots), dict(Counter(g["logic"] for g in roots)))

**이번 셀의 목표:** POL_GOV_006 사례: 원문과 조건 트리를 비교합니다 (DB 없이).
- 사용하는 개념: extracted vs verified
- 입력 데이터: DOC_GOV_006 원문, 인계 조건 트리
- 핵심 함수·매개변수: `str.count`
- 예상 출력: 원문에 무주택·소득·가구 0회, 어업인 5회
- 왜 실행하나요?: 그래프 조건을 무조건 믿으면 안 되는 이유를 데이터로 확인

In [ ]:
gov_doc = next(d for d in documents if d["document_id"] == "DOC_GOV_006")
print("문서:", gov_doc["title"])
print("원문 낱말 수:", {word: gov_doc["text"].count(word) for word in ("무주택", "소득", "가구", "어업인")})
gov_policy_node = next(n for n in nodes_by_label["Policy"] if n["policy_id"] == "POL_GOV_006")
print("Policy.verification_status:", gov_policy_node["verification_status"])
print("관계 review_status:", dict(Counter(r["review_status"] for r in handoff_relationships if r["source_id"] == "POL_GOV_006")))
offline_tree("GRP_POL_GOV_006_ROOT")

**해석 (의심, 확정 아님)**
- 그래프 조건: 연령 19~120 + **무주택 필수** + 소득 5구간 전부 OR + 가구 7종 OR
- 원문에는 무주택·소득·가구 언급이 없고, 실제 요건(어업인 등)은 그래프에 없음
- 원인 후보(인수인계서 §5-2): 정부24 조건 코드 변환 규칙. 해당 코드는 **장반석님 담당 파일**이며 수정하지 않았음
- 정부24 조건 트리가 있는 정책 82개 기준: 무주택 필수 승격 46개, 소득 전 구간 OR 45개, 가구 "해당사항없음" 누락 46개, 취업 조건 미반영 27개 (겹침 있음, 공식 원문 검증 아님)

**사용 원칙:** 자격 답변의 근거는 Qdrant 원문(`evidence`)을 우선한다. `suspected` 경고가 있는 정책은 "공고 확인 필요"로 안내한다.

**이번 셀의 목표:** (DB가 있으면) 데이터 품질 경고 함수를 실행합니다.
- 사용하는 개념: assess_policy_data_quality
- 입력 데이터: Chapter 9의 graph_context
- 핵심 함수·매개변수: `hs.assess_policy_data_quality(context)`
- 예상 출력: SUSPECTED_… 경고 코드
- 왜 실행하나요?: 검색 결과에 경고가 어떻게 붙는지 확인

In [ ]:
if graph_context and graph_context["policies"].get("POL_GOV_006"):
    quality = hs.assess_policy_data_quality(graph_context["policies"]["POL_GOV_006"])
    for warning in quality["warnings"]:
        print(warning["level"], warning["code"])
else:
    print("건너뜀 (Chapter 9에서 Neo4j 조회가 필요)")

### Chapter 13 정리
- 저장 무결성 통과 ≠ 의미 정확성
- UNKNOWN 155개는 "제한 없음"이 아니다.
- `extracted`는 자동 추출값, `verified`도 공식 원문 재검증이 필요할 수 있다.
- POL_GOV_006은 그래프 조건이 원문과 맞지 않는 의심 사례다.

**문제 13-1 (내 말로 설명하기):** 저장 무결성과 의미 정확성의 차이를 정책 예시로 설명하세요.

---
# Chapter 14. 발표·코드 리뷰 준비 (예상 질문과 쉬운 답변)

| 질문 | 쉬운 답변 |
|---|---|
| 왜 Qdrant인가? | 질문과 **의미가 비슷한 원문 조각**을 빠르게 찾고, `policy_id`·`document_id` 필터를 걸 수 있어서. 강의에서 다룬 도구이고 Docker로 로컬 실행이 쉽다. |
| 왜 Neo4j인가? | 정책–기관–지역–조건처럼 **관계가 많은 데이터**를 따라가기 쉽고, AND/OR 조건 계층을 그대로 저장할 수 있어서. |
| 왜 둘을 연결했나? | 벡터 검색은 "어떤 원문이 관련 있나"를, 그래프는 "그 정책의 기관·조건 구조"를 준다. `policy_id`로 묶으면 근거와 문맥을 함께 돌려줄 수 있다. |
| 왜 0.6B 모델인가? | 강의와 같은 모델이고, 노트북 CPU에서도 돌아가는 크기(639MB)라 **기준선**으로 썼다. 최종 모델은 비교 후 팀이 정한다. |
| 왜 B2 청킹인가? | 원문 경계를 지키면서 너무 짧은 조각만 합쳐서. **실험용 선택**이고, 평가 질문 세트로 A/B1/B2를 비교한 뒤 확정해야 한다. |
| 왜 107개만 적재했나? | 문서 5개로 **스모크 검증**(저장·검색·재적재·원문 일치)을 먼저 했다. 전체 246개 색인은 시연 범위 결정 후 진행한다. |
| 왜 Neo4j에는 237개 정책이 있나? | 그래프 인계 데이터는 전체가 한 번에 준비되어 있어 노드 1,034 / 관계 3,615를 모두 적재했다. 그래서 지금은 Qdrant 범위(5개)와 Neo4j 범위(237개)가 다르다. |
| 기존 인계 로더의 문제는? | 레이블 없이 `policy_id OR document_id …`로 매칭해서 관계가 3,615 → 11,316으로 부풀었다. 레이블 + 고유 ID로 고쳤다. |
| MERGE와 MATCH의 차이는? | MATCH는 **찾기만**(없으면 결과 없음), MERGE는 **찾고 없으면 만든다**(쓰기). 우리는 적재에만 MERGE, 조회는 MATCH. |
| UNKNOWN은 무슨 뜻? | 구조화된 조건이 없다는 뜻. "누구나 가능"이 아니라 "공고 원문 확인 필요". |
| 아직 미구현된 것은? | 전체 정책 색인, 평가 질문 세트 기반 성능 평가(Recall@K·MRR), 자격 판정 연결, LLM 답변 연결, BM25·재순위화, C안(LLM 섹션 청킹). |

### Chapter 14 연습
**문제 14-1:** 위 표를 보지 않고 "왜 Qdrant와 Neo4j를 연결했나?"를 30초 안에 설명해 보세요.
**문제 14-2:** "score 0.8이면 자격이 있는 건가요?"라는 질문에 어떻게 답할까요?

---
# Chapter 15. 정답 및 해설

<details><summary>Chapter 0</summary>

- 0-1: Qdrant는 질문과 의미가 비슷한 **원문 근거**를, Neo4j는 그 정책의 **관계·조건 구조**를 주기 때문에 둘을 함께 써야 "근거 + 문맥"을 돌려줄 수 있다.
- 0-2: 브랜치를 잘못 체크아웃했거나 최신 커밋을 받지 않은 경우. `git branch --show-current`, `git log`로 확인한다.
</details>

<details><summary>Chapter 1</summary>

- 1-1: `0.0` (두 벡터가 직각 → 방향이 전혀 다름)
- 1-2: `answer_policy_id = "POL_S2"` → `가상 주택기금`
</details>

<details><summary>Chapter 2</summary>

- 2-1: 다르다. 한 글자(공백 포함)라도 바뀌면 SHA-256 값이 완전히 달라진다.
- 2-2: 인계 관계에는 조건 구조 관계 3종(`HAS_ROOT_CONDITION_GROUP` 237, `CONTAINS_GROUP` 106, `CONTAINS_CONDITION` 735 = 1,078개)이 더 있다.
</details>

<details><summary>Chapter 3</summary>

- 3-1: B2 첫 청크의 `char_start`가 0이면 같다. 0이 아니면(앞부분에 공백·제외 구간이 있으면) 다를 수 있다. 직접 `chunks_by_strategy["B2"][0]["char_start"]`를 출력해 확인한다.
- 3-2: `strategy_name = "B1"`
</details>

<details><summary>Chapter 4</summary>

- 4-1: 내적과 같다. 코사인 = 내적 ÷ (1 × 1).
- 4-2: 원문(출처로 보여 줄 텍스트)은 바꾸지 않으면서, 검색에 도움이 되는 정책명·섹션명을 임베딩 입력에만 붙이기 위해. 규칙을 바꿔도 저장 원문은 그대로라 비교 실험이 쉽다.
</details>

<details><summary>Chapter 5</summary>

- 5-1: 2개 그대로. 같은 id는 덮어쓴다.
- 5-2: LangChain `QdrantVectorStore`는 metadata를 `metadata` 아래에 넣고, 우리는 payload 최상위에 직접 넣기 때문.
</details>

<details><summary>Chapter 6</summary>

- 6-1: 나올 수 없다. 필터가 `document_id = DOC_LH_001`인 point만 검색 대상으로 남긴다.
- 6-2: 근거 문장이 긴 목록의 한 줄이라 청크 전체 의미에 묻혔다는 가설. 개선 아이디어: 목록 항목 단위 분할, 키워드(BM25) 결합, 질문 재작성 (모두 미검증).
</details>

<details><summary>Chapter 7</summary>

- 7-1: 첫 번째는 `POL_S1` 정책 노드 1개, 두 번째는 빈 리스트(레이블이 Organization인 노드 중 id가 POL_S1인 것은 없음).
- 7-2: `label_name = "Organization"`
</details>

<details><summary>Chapter 8</summary>

- 8-1: 출발 후보가 Policy 1 + Document 3 + ConditionGroup 1 = 5개, 도착 1개 → 5개.
- 8-2: MERGE는 "찾고 없으면 생성"이지만, 동시에 여러 요청이 들어오는 등 경우에 DB 차원에서 같은 ID 노드가 둘 생기는 것을 **확실히 막고**, MERGE·MATCH 조회를 빠르게 하는 인덱스 역할도 하기 때문.
</details>

<details><summary>Chapter 9</summary>

- 9-1: 아니다. 구조화된 조건이 없을 뿐이므로 공고 원문 확인이 필요하다.
- 9-2: 자동 추출값이라 원문과 다를 수 있다(POL_GOV_006 사례). 답변 근거는 원문 청크를 우선한다.
</details>

<details><summary>Chapter 10</summary>

- 10-1: `MISSING_IN_GRAPH` (만들어 내지 않고 표시만 한다)
- 10-2: `k_value = 3`
</details>

<details><summary>Chapter 11~14</summary>

- 11-1: 강의는 그래프를 **도구로 독립 검색**(카테고리·언론사·기자)하고, 우리는 벡터 검색 결과의 policy_id로 **문맥을 보강**한다.
- 11-2: 전화번호·정책번호처럼 정확한 문자열은 의미 벡터보다 키워드 일치가 더 잘 잡을 수 있다(가설, 미검증).
- 12-1: 현재 검색은 의미 유사도만 보고 지역 필터·자격 판정을 하지 않기 때문. 지역 필터나 자격 확인 단계가 필요하다.
- 13-1: 예) DB의 POL_GOV_006 조건은 인계 파일과 똑같이 저장됐다(무결성 OK). 하지만 그 조건(무주택 필수 등)이 실제 공고 요건과 다를 수 있다(의미 정확성 미검증).
- 14-2: "아니요. score는 질문과 원문 조각이 의미상 얼마나 가까운지일 뿐이고, 자격은 판정하지 않습니다."
</details>

---
## 마무리: 실습 정리 (메모리 모드 Qdrant 닫기)
메모리 모드 Qdrant는 노트북을 끄면 사라집니다. 아래 셀은 열어 둔 연결을 정리합니다. (실제 DB에는 영향 없음)

In [ ]:
memory_client.close()
if QDRANT_OK:
    qdrant_client.close()
print("연결 정리 완료")